<a href="https://colab.research.google.com/github/vivek28n/Medical-RAG-Hallucination-Detection/blob/main/notebooks/Notebook_07_Self_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 07 — Self-Correction

## Objective

This notebook implements a self-correction mechanism for the medical RAG system.

The system evaluates the generated answer using evidence-based hallucination detection
and confidence scoring. If the answer is considered weak, the system re-checks the
retrieved evidence and generates a revised answer grounded only in the available sources.

## Pipeline

Question
→ RAG Answer
→ Hallucination Detection
→ Confidence Scoring
→ Self-Correction when required
→ Re-evaluation
→ Final Answer

## Goal

Reduce unsupported or potentially hallucinated claims by allowing the system to
critically re-evaluate and revise its own generated answer.

In [1]:
import re
import numpy as np
from google import genai
from google.colab import userdata

In [2]:
API_KEY = userdata.get("Vivek28n")

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3.8-flash"

In [3]:
def should_self_correct(confidence_score, decision):
    if decision in ["POTENTIAL HALLUCINATION", "CONTRADICTED"]:
        return True

    if confidence_score < 0.60:
        return True

    return False

In [4]:
def self_correct_answer(question, answer, retrieved_documents):
    evidence_text = "\n\n".join(
        [
            f"[Source: {doc.get('source', 'Unknown')} | Page: {doc.get('page', 'Unknown')}]\n"
            f"{doc['text']}"
            for doc in retrieved_documents
        ]
    )

    prompt = f"""
You are a medical evidence verification assistant.

Your task is to review and correct an AI-generated answer using ONLY the
provided evidence.

Question:
{question}

Original Answer:
{answer}

Retrieved Evidence:
{evidence_text}

Instructions:
1. Keep only claims that are supported by the retrieved evidence.
2. Remove unsupported or speculative claims.
3. Correct any claim that conflicts with the evidence.
4. Do not add medical facts from your own knowledge.
5. If the evidence is insufficient, explicitly say that there is not enough
   evidence to answer that part.
6. Keep the answer concise and medically grounded.
7. Include source and page references for supported claims.

Return only the corrected answer.
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text.strip()

In [5]:
def self_correct_answer(question, answer, retrieved_documents):
    evidence_text = "\n\n".join(
        [
            f"[Source: {doc.get('source', 'Unknown')} | Page: {doc.get('page', 'Unknown')}]\n"
            f"{doc['text']}"
            for doc in retrieved_documents
        ]
    )

    prompt = f"""
You are a medical evidence verification assistant.

Your task is to review and correct an AI-generated answer using ONLY the
provided evidence.

Question:
{question}

Original Answer:
{answer}

Retrieved Evidence:
{evidence_text}

Instructions:
1. Keep only claims supported by the retrieved evidence.
2. Remove unsupported or speculative claims.
3. Correct claims that conflict with the evidence.
4. Do not add medical facts from your own knowledge.
5. If the evidence is insufficient, explicitly say so.
6. Keep the answer concise and medically grounded.
7. Include source and page references for supported claims.

Return only the corrected answer.
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text.strip()

In [6]:
print(should_self_correct(0.85, "SUPPORTED"))
print(should_self_correct(0.54, "SUPPORTED"))
print(should_self_correct(0.70, "POTENTIAL HALLUCINATION"))

False
True
True


In [7]:
# ============================================
# RESTORE RAG RETRIEVAL PIPELINE
# ============================================

import os
import nbformat

PROJECT_DIR = "/content/Medical-RAG-Hallucination-Detection"

# Clone repository if it is not already available
if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/vivek28n/Medical-RAG-Hallucination-Detection.git

%cd /content/Medical-RAG-Hallucination-Detection

print("Project directory:", os.getcwd())

Cloning into 'Medical-RAG-Hallucination-Detection'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 117 (delta 65), reused 34 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 1.48 MiB | 10.39 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/content/Medical-RAG-Hallucination-Detection
Project directory: /content/Medical-RAG-Hallucination-Detection


In [8]:
# ============================================
# RESTORE NOTEBOOK 3 PIPELINE
# ============================================

NOTEBOOK3 = os.path.join(
    PROJECT_DIR,
    "notebooks",
    "Notebook_03_PDF_Text_Extraction.ipynb"
)

with open(NOTEBOOK3, "r", encoding="utf-8") as f:
    nb3 = nbformat.read(f, as_version=4)

print("Notebook 3 loaded successfully.")
print("Total cells:", len(nb3.cells))

Notebook 3 loaded successfully.
Total cells: 28


In [9]:
# ============================================
# INSPECT NOTEBOOK 3 CELLS
# ============================================

for i, cell in enumerate(nb3.cells):
    print(f"\n========== Cell {i + 1} | {cell.cell_type} ==========")
    print(cell.source[:500])


========== Cell 1 | code ==========
!git clone https://github.com/vivek28n/Medical-RAG-Hallucination-Detection.git

========== Cell 2 | code ==========
%cd /content/Medical-RAG-Hallucination-Detection

========== Cell 3 | code ==========
import os

print("Dataset:", os.path.exists("dataset"))
print("Raw:", os.path.exists("dataset/raw"))
print("Files:", os.listdir("dataset/raw"))

========== Cell 4 | code ==========
!git pull origin main

========== Cell 5 | code ==========
import os
print(os.listdir("dataset/raw"))

========== Cell 6 | code ==========
!pip install -q pymupdf

========== Cell 7 | code ==========
import os

print(os.getcwd())
print(os.listdir("dataset/raw"))

========== Cell 8 | code ==========
import fitz

pdf_path = "dataset/raw/niddk_guiding_principles_diabetes.pdf"

doc = fitz.open(pdf_path)

print("Number of pages:", len(doc))

========== Cell 9 | code ==========
text = ""

for page in doc:
    text += page.get_text()

print("Characters extracted:", len(text))
prin

In [10]:
!pip install -q pymupdf faiss-cpu langchain-text-splitters sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 56.9 MB/s eta 0:00:00


In [11]:
import fitz
import faiss

print("✅ PyMuPDF installed")
print("✅ FAISS installed")

✅ PyMuPDF installed
✅ FAISS installed


In [12]:
# ============================================
# BUILD RETRIEVAL DATA FOR NOTEBOOK 7
# ============================================

import fitz
import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = "dataset/raw/niddk_guiding_principles_diabetes.pdf"

doc = fitz.open(PDF_PATH)

pages = []

for page_number, page in enumerate(doc, start=1):
    page_text = page.get_text().strip()

    pages.append({
        "page": page_number,
        "text": page_text
    })

print("Total pages:", len(pages))


def clean_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    text = "\n".join(
        line.strip() for line in text.splitlines()
    )
    return text.strip()


for page in pages:
    page["clean_text"] = clean_text(page["text"])

print("Text cleaning completed.")


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for page in pages:
    page_chunks = splitter.split_text(page["clean_text"])

    for chunk_id, chunk in enumerate(page_chunks):
        chunks.append({
            "chunk_id": f"page_{page['page']}_chunk_{chunk_id}",
            "page": page["page"],
            "text": chunk
        })

print("Total chunks:", len(chunks))


embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

embedding_array = np.array(
    embeddings
).astype("float32")

print("Embedding shape:", embedding_array.shape)


dimension = embedding_array.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_array)

print("Total vectors in FAISS:", index.ntotal)

Total pages: 83
Text cleaning completed.
Total chunks: 277


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedding shape: (277, 384)
Total vectors in FAISS: 277


In [13]:
# ============================================
# DOCUMENT RETRIEVAL FUNCTION
# ============================================

def retrieve_documents(question, top_k=5):
    """
    Retrieve the most relevant medical chunks
    from the FAISS index.
    """

    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):

        results.append({
            "rank": rank,
            "chunk_id": chunks[idx]["chunk_id"],
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "distance": float(distances[0][rank - 1])
        })

    return results

In [14]:
question = "What are the risk factors for diabetes?"

retrieved_documents = retrieve_documents(
    question,
    top_k=5
)

print("Retrieved documents:", len(retrieved_documents))

for doc in retrieved_documents:
    print(
        f"\nRank: {doc['rank']}"
        f"\nPage: {doc['page']}"
        f"\nDistance: {doc['distance']:.4f}"
    )
    print(doc["text"][:300])
    print("-" * 60)

Retrieved documents: 5

Rank: 1
Page: 5
Distance: 0.6568
5
INTRODUCTION
The diabetes problem
Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million
who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes
also increases the risk of cardiovascular disease, cancer, and dementia a
------------------------------------------------------------

Rank: 2
Page: 9
Distance: 0.6770
9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Risk of type 2 diabetes increases with age
and is strongly associated with overweight or
obesity—body mass index (BMI) ≥ 25 kg/m2
(≥ 23 kg/m2 for Asian Americans5)

Additional risk factors include
1.

2.

3.
4.
5.
Family
------------------------------------------------------------

Rank: 3
Page: 5
Distance: 0.7260
by the Centers for Disease Control and Prevention (CDC) and other organizations.
Proper nutrition and physical activity are the cornerstones of treatme

In [16]:
import time

def self_correct_answer_with_retry(
    question,
    answer,
    retrieved_documents,
    max_retries=3
):
    for attempt in range(max_retries):
        try:
            return self_correct_answer(
                question,
                answer,
                retrieved_documents
            )

        except Exception as e:
            error_message = str(e)

            if "503" in error_message or "UNAVAILABLE" in error_message:
                print(
                    f"Gemini temporarily unavailable. "
                    f"Retry {attempt + 1}/{max_retries}..."
                )

                if attempt < max_retries - 1:
                    time.sleep(5)
                else:
                    raise

            else:
                raise

In [18]:
# ============================================
# DOCUMENT RETRIEVAL FUNCTION
# ============================================

def retrieve_documents(question, top_k=5):
    """
    Retrieve the most relevant medical chunks
    from the FAISS index.
    """

    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):

        results.append({
            "rank": rank,
            "chunk_id": chunks[idx]["chunk_id"],
            "source": "NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes",
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "distance": float(distances[0][rank - 1])
        })

    return results

In [19]:
question = "What are the risk factors for diabetes?"

retrieved_documents = retrieve_documents(
    question,
    top_k=5
)

print("Retrieved documents:", len(retrieved_documents))

for doc in retrieved_documents:
    print(
        f"\nRank: {doc['rank']}"
        f"\nSource: {doc['source']}"
        f"\nPage: {doc['page']}"
        f"\nDistance: {doc['distance']:.4f}"
    )
    print(doc["text"][:200])
    print("-" * 60)

Retrieved documents: 5

Rank: 1
Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes
Page: 5
Distance: 0.6568
5
INTRODUCTION
The diabetes problem
Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million
who are undiagnosed.1 A major cause of blindness, renal failure
------------------------------------------------------------

Rank: 2
Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes
Page: 9
Distance: 0.6770
9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Risk of type 2 diabetes increases with age
and is strongly associated with overweight or
obesity—body mass index (BMI) ≥
------------------------------------------------------------

Rank: 3
Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes
Page: 5
Distance: 0.7260
by the Centers for Disease Control and Prevention (CDC) and other organizations.
Proper nutritio

In [20]:
import time

def self_correct_answer_with_retry(
    question,
    answer,
    retrieved_documents,
    max_retries=3
):
    for attempt in range(max_retries):
        try:
            return self_correct_answer(
                question,
                answer,
                retrieved_documents
            )

        except Exception as e:
            error_message = str(e)

            if "503" in error_message or "UNAVAILABLE" in error_message:
                print(
                    f"Gemini temporarily unavailable. "
                    f"Retry {attempt + 1}/{max_retries}..."
                )

                if attempt < max_retries - 1:
                    time.sleep(5)
                else:
                    raise

            else:
                raise

In [22]:
# ============================================
# SELF-CORRECTION DECISION LOOP
# ============================================

def run_self_correction_loop(
    question,
    answer,
    retrieved_documents
):
    """
    Analyze an answer and automatically self-correct
    it when hallucination risk is detected.
    """

    # Step 1: Analyze original answer
    initial_analysis = analyze_answer(
        answer,
        retrieved_documents
    )

    initial_confidence = calculate_confidence(
        initial_analysis["semantic_similarity"],
        initial_analysis["nli_entailment"],
        retrieved_documents,
        answer
    )

    initial_decision = initial_analysis["decision"]

    # Step 2: Decide whether correction is needed
    correction_needed = should_self_correct(
        initial_confidence,
        initial_decision
    )

    # Step 3: Keep original if answer is already good
    if not correction_needed:
        return {
            "original_answer": answer,
            "final_answer": answer,
            "initial_analysis": initial_analysis,
            "initial_confidence": initial_confidence,
            "correction_applied": False
        }

    # Step 4: Correct the answer
    corrected_answer = self_correct_answer_with_retry(
        question,
        answer,
        retrieved_documents
    )

    # Step 5: Re-analyze corrected answer
    corrected_analysis = analyze_answer(
        corrected_answer,
        retrieved_documents
    )

    corrected_confidence = calculate_confidence(
        corrected_analysis["semantic_similarity"],
        corrected_analysis["nli_entailment"],
        retrieved_documents,
        corrected_answer
    )

    # Step 6: Return complete result
    return {
        "original_answer": answer,
        "final_answer": corrected_answer,
        "initial_analysis": initial_analysis,
        "initial_confidence": initial_confidence,
        "corrected_analysis": corrected_analysis,
        "corrected_confidence": corrected_confidence,
        "correction_applied": True
    }

In [23]:
# ============================================
# RESTORE HALLUCINATION ANALYSIS
# ============================================

def semantic_similarity(answer, retrieved_documents):
    answer_embedding = embedding_model.encode(
        answer,
        normalize_embeddings=True
    )

    document_texts = [
        doc["text"] for doc in retrieved_documents
    ]

    document_embeddings = embedding_model.encode(
        document_texts,
        normalize_embeddings=True
    )

    similarities = np.dot(
        document_embeddings,
        answer_embedding
    )

    max_similarity = float(
        np.max(similarities)
    )

    return max_similarity, similarities

In [24]:
# ============================================
# NLI SCORING
# ============================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

NLI_MODEL = "cross-encoder/nli-deberta-v3-base"

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL
)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL
)

nli_model.eval()


def nli_score(premise, hypothesis):

    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    contradiction_score = float(probabilities[0])
    entailment_score = float(probabilities[1])
    neutral_score = float(probabilities[2])

    return (
        entailment_score,
        neutral_score,
        contradiction_score
    )

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [25]:
def check_nli_against_documents(
    answer,
    retrieved_documents
):
    results = []

    for rank, doc in enumerate(
        retrieved_documents,
        start=1
    ):

        entailment, neutral, contradiction = nli_score(
            doc["text"],
            answer
        )

        results.append({
            "rank": rank,
            "page": doc.get("page"),
            "entailment": entailment,
            "neutral": neutral,
            "contradiction": contradiction
        })

    return results

In [26]:
def get_max_entailment(nli_results):
    return max(
        result["entailment"]
        for result in nli_results
    )


def calculate_support_score(
    similarity_score,
    entailment_score
):
    return (
        0.5 * similarity_score
        + 0.5 * entailment_score
    )


SUPPORT_THRESHOLD = 0.60
CONTRADICTION_THRESHOLD = 0.50


def detect_hallucination(
    support_score,
    nli_results,
    support_threshold=SUPPORT_THRESHOLD,
    contradiction_threshold=CONTRADICTION_THRESHOLD
):

    max_contradiction = max(
        result["contradiction"]
        for result in nli_results
    )

    if max_contradiction >= contradiction_threshold:
        return "CONTRADICTED"

    elif support_score >= support_threshold:
        return "SUPPORTED"

    else:
        return "POTENTIAL HALLUCINATION"


def analyze_answer(
    answer,
    retrieved_documents
):

    similarity_score, similarities = semantic_similarity(
        answer,
        retrieved_documents
    )

    nli_results = check_nli_against_documents(
        answer,
        retrieved_documents
    )

    entailment_score = get_max_entailment(
        nli_results
    )

    support_score = calculate_support_score(
        similarity_score,
        entailment_score
    )

    decision = detect_hallucination(
        support_score,
        nli_results
    )

    return {
        "semantic_similarity": similarity_score,
        "nli_entailment": entailment_score,
        "support_score": support_score,
        "decision": decision,
        "nli_results": nli_results
    }

In [27]:
# ============================================
# CONFIDENCE SCORING
# ============================================

def calculate_retrieval_quality(
    question,
    retrieved_documents
):
    """
    Calculate retrieval quality from the similarity
    between the question and retrieved documents.
    """

    query_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    )

    document_texts = [
        doc["text"] for doc in retrieved_documents
    ]

    document_embeddings = embedding_model.encode(
        document_texts,
        normalize_embeddings=True
    )

    similarities = np.dot(
        document_embeddings,
        query_embedding
    )

    return float(np.mean(similarities))


def calculate_consistency(
    answer,
    retrieved_documents
):
    """
    Estimate claim-level consistency between the answer
    and retrieved evidence.
    """

    # Remove source citations and bullet formatting
    cleaned_answer = re.sub(
        r"\[Source:.*?\| Page:.*?\]",
        "",
        answer
    )

    cleaned_answer = re.sub(
        r"[*•]",
        "",
        cleaned_answer
    )

    # Split answer into claims/sentences
    claims = [
        claim.strip()
        for claim in re.split(
            r"[.!?]\s+",
            cleaned_answer
        )
        if claim.strip()
    ]

    if not claims:
        return 0.0

    evidence_sentences = []

    for doc in retrieved_documents:
        sentences = re.split(
            r"(?<=[.!?])\s+",
            doc["text"]
        )

        evidence_sentences.extend(
            sentence.strip()
            for sentence in sentences
            if sentence.strip()
        )

    supported_claims = 0

    for claim in claims:

        claim_embedding = embedding_model.encode(
            claim,
            normalize_embeddings=True
        )

        evidence_embeddings = embedding_model.encode(
            evidence_sentences,
            normalize_embeddings=True
        )

        similarities = np.dot(
            evidence_embeddings,
            claim_embedding
        )

        best_index = int(
            np.argmax(similarities)
        )

        best_similarity = float(
            similarities[best_index]
        )

        entailment, neutral, contradiction = nli_score(
            evidence_sentences[best_index],
            claim
        )

        claim_support = (
            0.5 * best_similarity
            + 0.5 * entailment
        )

        if (
            best_similarity >= 0.40
            and claim_support >= 0.50
        ):
            supported_claims += 1

    return supported_claims / len(claims)


def calculate_confidence(
    semantic_similarity_score,
    nli_entailment_score,
    retrieved_documents,
    answer
):
    """
    Calculate overall confidence score.
    """

    retrieval_quality = calculate_retrieval_quality(
        question,
        retrieved_documents
    )

    consistency = calculate_consistency(
        answer,
        retrieved_documents
    )

    confidence_score = (
        0.30 * semantic_similarity_score
        + 0.30 * nli_entailment_score
        + 0.20 * retrieval_quality
        + 0.20 * consistency
    )

    return float(confidence_score)

In [28]:
# ============================================
# FINAL CONFIDENCE FUNCTION
# ============================================

def calculate_confidence(
    question,
    semantic_similarity_score,
    nli_entailment_score,
    retrieved_documents,
    answer
):
    """
    Calculate the overall confidence score using:

    30% Semantic Similarity
    30% NLI Entailment
    20% Retrieval Quality
    20% Claim Consistency
    """

    retrieval_quality = calculate_retrieval_quality(
        question,
        retrieved_documents
    )

    consistency = calculate_consistency(
        answer,
        retrieved_documents
    )

    confidence_score = (
        0.30 * semantic_similarity_score
        + 0.30 * nli_entailment_score
        + 0.20 * retrieval_quality
        + 0.20 * consistency
    )

    return {
        "confidence_score": float(confidence_score),
        "retrieval_quality": float(retrieval_quality),
        "consistency": float(consistency)
    }

In [29]:
# ============================================
# COMPLETE SELF-CORRECTION LOOP
# ============================================

def run_self_correction_loop(
    question,
    answer,
    retrieved_documents
):
    """
    Complete pipeline:

    Answer
      ↓
    Hallucination Detection
      ↓
    Confidence Scoring
      ↓
    Self-Correction if needed
      ↓
    Re-analysis
      ↓
    Final Answer
    """

    # ----------------------------------------
    # STEP 1: Analyze original answer
    # ----------------------------------------

    initial_analysis = analyze_answer(
        answer,
        retrieved_documents
    )

    initial_confidence_data = calculate_confidence(
        question,
        initial_analysis["semantic_similarity"],
        initial_analysis["nli_entailment"],
        retrieved_documents,
        answer
    )

    initial_confidence = initial_confidence_data[
        "confidence_score"
    ]

    initial_decision = initial_analysis["decision"]

    # ----------------------------------------
    # STEP 2: Decide whether correction needed
    # ----------------------------------------

    correction_needed = should_self_correct(
        initial_confidence,
        initial_decision
    )

    # ----------------------------------------
    # STEP 3: No correction needed
    # ----------------------------------------

    if not correction_needed:

        return {
            "original_answer": answer,
            "final_answer": answer,

            "initial_decision": initial_decision,

            "initial_confidence":
                initial_confidence,

            "initial_retrieval_quality":
                initial_confidence_data["retrieval_quality"],

            "initial_consistency":
                initial_confidence_data["consistency"],

            "correction_applied": False
        }

    # ----------------------------------------
    # STEP 4: Self-correct
    # ----------------------------------------

    corrected_answer = self_correct_answer_with_retry(
        question,
        answer,
        retrieved_documents
    )

    # ----------------------------------------
    # STEP 5: Re-analyze corrected answer
    # ----------------------------------------

    corrected_analysis = analyze_answer(
        corrected_answer,
        retrieved_documents
    )

    corrected_confidence_data = calculate_confidence(
        question,
        corrected_analysis["semantic_similarity"],
        corrected_analysis["nli_entailment"],
        retrieved_documents,
        corrected_answer
    )

    corrected_confidence = corrected_confidence_data[
        "confidence_score"
    ]

    # ----------------------------------------
    # STEP 6: Return complete result
    # ----------------------------------------

    return {
        "original_answer": answer,

        "final_answer": corrected_answer,

        "initial_decision": initial_decision,

        "initial_confidence":
            initial_confidence,

        "initial_retrieval_quality":
            initial_confidence_data["retrieval_quality"],

        "initial_consistency":
            initial_confidence_data["consistency"],

        "corrected_decision":
            corrected_analysis["decision"],

        "corrected_confidence":
            corrected_confidence,

        "corrected_retrieval_quality":
            corrected_confidence_data["retrieval_quality"],

        "corrected_consistency":
            corrected_confidence_data["consistency"],

        "correction_applied": True
    }

In [31]:
# ============================================
# OPTIMIZED CONSISTENCY CHECK
# ============================================

def calculate_consistency(
    answer,
    retrieved_documents
):
    """
    Estimate claim-level consistency between
    answer and retrieved evidence.
    """

    cleaned_answer = re.sub(
        r"\[Source:.*?\| Page:.*?\]",
        "",
        answer
    )

    cleaned_answer = re.sub(
        r"[*•]",
        "",
        cleaned_answer
    )

    claims = [
        claim.strip()
        for claim in re.split(
            r"[.!?]\s+",
            cleaned_answer
        )
        if claim.strip()
    ]

    if not claims:
        return 0.0

    evidence_sentences = []

    for doc in retrieved_documents:
        sentences = re.split(
            r"(?<=[.!?])\s+",
            doc["text"]
        )

        evidence_sentences.extend(
            sentence.strip()
            for sentence in sentences
            if sentence.strip()
        )

    # Embed evidence only once
    evidence_embeddings = embedding_model.encode(
        evidence_sentences,
        normalize_embeddings=True
    )

    supported_claims = 0

    for i, claim in enumerate(claims, start=1):

        print(
            f"Checking claim {i}/{len(claims)}..."
        )

        claim_embedding = embedding_model.encode(
            claim,
            normalize_embeddings=True
        )

        similarities = np.dot(
            evidence_embeddings,
            claim_embedding
        )

        best_index = int(
            np.argmax(similarities)
        )

        best_similarity = float(
            similarities[best_index]
        )

        entailment, neutral, contradiction = nli_score(
            evidence_sentences[best_index],
            claim
        )

        claim_support = (
            0.5 * best_similarity
            + 0.5 * entailment
        )

        if (
            best_similarity >= 0.40
            and claim_support >= 0.50
        ):
            supported_claims += 1

    consistency = (
        supported_claims / len(claims)
    )

    print(
        f"Supported claims: "
        f"{supported_claims}/{len(claims)}"
    )

    return consistency

In [32]:
# ============================================
# CLOSED-LOOP TEST
# ============================================

test_question = "What are the risk factors for diabetes?"

test_answer = """
Diabetes risk factors include obesity, family history,
physical inactivity, smoking, vitamin D deficiency,
and excessive salt consumption.
"""

test_documents = retrieve_documents(
    test_question,
    top_k=5
)

print("Starting closed-loop test...")

result = run_self_correction_loop(
    test_question,
    test_answer,
    test_documents
)

print("\n" + "=" * 70)

print("CORRECTION APPLIED:")
print(result["correction_applied"])

print("\nINITIAL DECISION:")
print(result["initial_decision"])

print("\nINITIAL CONFIDENCE:")
print(round(result["initial_confidence"], 4))

if result["correction_applied"]:

    print("\nCORRECTED DECISION:")
    print(result["corrected_decision"])

    print("\nCORRECTED CONFIDENCE:")
    print(round(result["corrected_confidence"], 4))

    print("\nCONFIDENCE CHANGE:")
    print(
        round(
            result["corrected_confidence"]
            - result["initial_confidence"],
            4
        )
    )

print("\n" + "=" * 70)

print("FINAL ANSWER:")
print(result["final_answer"])

Starting closed-loop test...
Checking claim 1/1...
Supported claims: 0/1
Gemini temporarily unavailable. Retry 1/3...
Gemini temporarily unavailable. Retry 2/3...
Checking claim 1/1...
Supported claims: 0/1

CORRECTION APPLIED:
True

INITIAL DECISION:
POTENTIAL HALLUCINATION

INITIAL CONFIDENCE:
0.3363

CORRECTED DECISION:
SUPPORTED

CORRECTED CONFIDENCE:
0.5851

CONFIDENCE CHANGE:
0.2488

FINAL ANSWER:
Based on the provided evidence, risk factors for type 2 diabetes include:

- **Overweight or obesity** (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans) [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
- **Increasing age** [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
- **Family history of diabetes** (parent or sibling) [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
- **Physical inactivity** [Source: NIDDK Guiding Principles for the Care 

In [33]:
# ============================================
# IMPROVED CLAIM EXTRACTION
# ============================================

def extract_claims(answer):
    """
    Extract individual claims from an answer.

    Handles:
    - bullet points
    - numbered lists
    - normal sentences
    """

    cleaned_answer = re.sub(
        r"\[Source:.*?\| Page:.*?\]",
        "",
        answer
    )

    lines = cleaned_answer.splitlines()

    claims = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Remove markdown bullets
        line = re.sub(
            r"^[\*\-•]\s*",
            "",
            line
        )

        # Remove numbered list prefixes
        line = re.sub(
            r"^\d+[\.\)]\s*",
            "",
            line
        )

        # Ignore headings / notes
        if line.lower().startswith("based on the"):
            continue

        if line.lower().startswith("(note:"):
            continue

        if len(line) > 20:
            claims.append(line)

    # If no bullet/list structure exists,
    # fall back to sentence splitting
    if not claims:
        claims = [
            claim.strip()
            for claim in re.split(
                r"[.!?]\s+",
                cleaned_answer
            )
            if len(claim.strip()) > 20
        ]

    return claims

In [34]:
claims = extract_claims(test_answer)

print("Number of claims:", len(claims))

for i, claim in enumerate(claims, start=1):
    print(f"\nClaim {i}:")
    print(claim)

Number of claims: 3

Claim 1:
Diabetes risk factors include obesity, family history,

Claim 2:
physical inactivity, smoking, vitamin D deficiency,

Claim 3:
and excessive salt consumption.


In [35]:
import re


def extract_claims(answer):
    """
    Extract individual factual claims from an answer.

    Handles:
    - Bullet points
    - Numbered lists
    - New-line separated claims
    - Comma-separated risk-factor lists
    """

    if not answer or not answer.strip():
        return []

    # -----------------------------------------
    # Step 1: Clean answer
    # -----------------------------------------

    text = answer.strip()

    # Remove markdown bullets / numbering
    text = re.sub(
        r"(?m)^\s*[-*•]\s*",
        "",
        text
    )

    text = re.sub(
        r"(?m)^\s*\d+[\.\)]\s*",
        "",
        text
    )

    # -----------------------------------------
    # Step 2: Split into lines
    # -----------------------------------------

    lines = [
        line.strip()
        for line in text.split("\n")
        if line.strip()
    ]

    claims = []

    # -----------------------------------------
    # Step 3: Process each line
    # -----------------------------------------

    for line in lines:

        # Remove citation markers if present
        line = re.sub(
            r"\[[^\]]+\]",
            "",
            line
        ).strip()

        if not line:
            continue

        # -------------------------------------
        # Detect "risk factors include..."
        # -------------------------------------

        lower_line = line.lower()

        if (
            "risk factors include" in lower_line
            and "," in line
        ):

            # Take everything after "include"
            match = re.search(
                r"risk factors include\s+(.+)",
                line,
                flags=re.IGNORECASE
            )

            if match:

                factor_text = match.group(1).strip()

                # Remove final punctuation
                factor_text = factor_text.rstrip(". ")

                # Split individual factors
                factors = [
                    factor.strip()
                    for factor in factor_text.split(",")
                    if factor.strip()
                ]

                for factor in factors:

                    # Convert each factor into a factual claim
                    claims.append(
                        f"{factor} is a risk factor for type 2 diabetes."
                    )

                continue

        # -------------------------------------
        # Normal sentence
        # -------------------------------------

        sentences = re.split(
            r"(?<=[.!?])\s+",
            line
        )

        for sentence in sentences:

            sentence = sentence.strip()

            if len(sentence) > 10:
                claims.append(sentence)

    # -----------------------------------------
    # Step 4: Remove duplicates
    # -----------------------------------------

    unique_claims = []

    seen = set()

    for claim in claims:

        normalized = claim.lower().strip()

        if normalized not in seen:
            seen.add(normalized)
            unique_claims.append(claim)

    return unique_claims

In [36]:
test_answer = """
Diabetes risk factors include obesity, family history,
physical inactivity, smoking, vitamin D deficiency,
and excessive salt consumption.
"""

test_claims = extract_claims(test_answer)

print("Total claims:", len(test_claims))

for i, claim in enumerate(test_claims, start=1):
    print(f"{i}. {claim}")

Total claims: 4
1. obesity is a risk factor for type 2 diabetes.
2. family history is a risk factor for type 2 diabetes.
3. physical inactivity, smoking, vitamin D deficiency,
4. and excessive salt consumption.


In [37]:
claims = extract_claims(test_answer)

print("Number of claims:", len(claims))

for i, claim in enumerate(claims, start=1):
    print(f"\nClaim {i}:")
    print(claim)

Number of claims: 4

Claim 1:
obesity is a risk factor for type 2 diabetes.

Claim 2:
family history is a risk factor for type 2 diabetes.

Claim 3:
physical inactivity, smoking, vitamin D deficiency,

Claim 4:
and excessive salt consumption.


In [38]:
corrected_claims = extract_claims(
    result["final_answer"]
)

print("Number of claims:", len(corrected_claims))

for i, claim in enumerate(
    corrected_claims,
    start=1
):
    print(f"\nClaim {i}:")
    print(claim)

Number of claims: 10

Claim 1:
Based on the provided evidence, risk factors for type 2 diabetes include:

Claim 2:
**Overweight or obesity** (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans)

Claim 3:
**Increasing age**

Claim 4:
**Family history of diabetes** (parent or sibling)

Claim 5:
**Physical inactivity**

Claim 6:
**Being a member of a high-risk population** (African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, Pacific Islander American)

Claim 7:
**History of gestational diabetes mellitus (GDM)**

Claim 8:
**Hypertension**

Claim 9:
**Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day)** as emerging risk factors

Claim 10:
(Note: Claims that smoking, vitamin D deficiency, and excessive salt consumption are risk factors for diabetes have been removed because they are not supported by the provided evidence.)*


In [39]:
# ============================================
# CONSISTENCY TEST
# ============================================

consistency_score = calculate_consistency(
    result["final_answer"],
    test_documents
)

print("\nConsistency Score:")
print(round(consistency_score, 4))

Checking claim 1/1...
Supported claims: 0/1

Consistency Score:
0.0


In [40]:
# ============================================
# DEBUG CLAIM EXTRACTION
# ============================================

print("FINAL ANSWER:")
print(result["final_answer"])

print("\n" + "=" * 70)

debug_claims = extract_claims(
    result["final_answer"]
)

print("NUMBER OF EXTRACTED CLAIMS:")
print(len(debug_claims))

print("\n" + "=" * 70)

for i, claim in enumerate(
    debug_claims,
    start=1
):
    print(f"\nCLAIM {i}:")
    print(repr(claim))

FINAL ANSWER:
Based on the provided evidence, risk factors for type 2 diabetes include:

- **Overweight or obesity** (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans) [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
- **Increasing age** [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
- **Family history of diabetes** (parent or sibling) [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
- **Physical inactivity** [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
- **Being a member of a high-risk population** (African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, Pacific Islander American) [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
- **History of gestational diabetes mellitus (GDM)** [Source: NIDDK Guiding Principles f

In [41]:
# ============================================
# CURRENT FINAL ANSWER CONSISTENCY TEST
# ============================================

current_answer = result["final_answer"]

consistency_score = calculate_consistency(
    current_answer,
    test_documents
)

print("\n" + "=" * 70)
print("FINAL CONSISTENCY SCORE:")
print(round(consistency_score, 4))

Checking claim 1/1...
Supported claims: 0/1

FINAL CONSISTENCY SCORE:
0.0


In [42]:
# ============================================
# FINAL CLAIM-LEVEL CONSISTENCY
# ============================================

def calculate_consistency(
    answer,
    retrieved_documents
):
    """
    Calculate claim-level consistency between
    an answer and retrieved evidence.
    """

    # ----------------------------------------
    # Extract claims directly
    # ----------------------------------------

    cleaned_answer = re.sub(
        r"\[Source:.*?\| Page:.*?\]",
        "",
        answer
    ).strip()

    claims = []

    bullet_matches = re.findall(
        r"(?m)^[\*\-•]\s+(.+?)(?=\n[\*\-•]\s+|\Z)",
        cleaned_answer,
        flags=re.DOTALL
    )

    if bullet_matches:

        for claim in bullet_matches:

            claim = re.sub(
                r"\s+",
                " ",
                claim
            ).strip()

            # Remove trailing note
            claim = re.sub(
                r"\s*\*\(Note:.*$",
                "",
                claim
            ).strip()

            if len(claim) > 20:
                claims.append(claim)

    else:

        paragraphs = re.split(
            r"\n\s*\n",
            cleaned_answer
        )

        for paragraph in paragraphs:

            paragraph = re.sub(
                r"\s+",
                " ",
                paragraph
            ).strip()

            if len(paragraph) > 20:
                claims.append(paragraph)

    print("Total claims detected:", len(claims))

    if not claims:
        return 0.0

    # ----------------------------------------
    # Prepare evidence
    # ----------------------------------------

    evidence_sentences = []

    for doc in retrieved_documents:

        sentences = re.split(
            r"(?<=[.!?])\s+",
            doc["text"]
        )

        for sentence in sentences:

            sentence = sentence.strip()

            if len(sentence) > 20:
                evidence_sentences.append(
                    sentence
                )

    if not evidence_sentences:
        return 0.0

    # ----------------------------------------
    # Embed evidence once
    # ----------------------------------------

    evidence_embeddings = embedding_model.encode(
        evidence_sentences,
        normalize_embeddings=True
    )

    supported_claims = 0

    # ----------------------------------------
    # Verify each claim
    # ----------------------------------------

    for i, claim in enumerate(
        claims,
        start=1
    ):

        print(
            f"Checking claim {i}/{len(claims)}..."
        )

        clean_claim = re.sub(
            r"[*]",
            "",
            claim
        ).strip()

        claim_embedding = embedding_model.encode(
            clean_claim,
            normalize_embeddings=True
        )

        similarities = np.dot(
            evidence_embeddings,
            claim_embedding
        )

        best_index = int(
            np.argmax(similarities)
        )

        best_similarity = float(
            similarities[best_index]
        )

        best_evidence = (
            evidence_sentences[best_index]
        )

        entailment, neutral, contradiction = nli_score(
            best_evidence,
            clean_claim
        )

        claim_support = (
            0.5 * best_similarity
            + 0.5 * entailment
        )

        if (
            best_similarity >= 0.40
            and entailment >= 0.50
            and claim_support >= 0.50
        ):
            supported_claims += 1

    consistency = (
        supported_claims / len(claims)
    )

    print(
        f"\nSupported claims: "
        f"{supported_claims}/{len(claims)}"
    )

    return float(consistency)

In [43]:
current_answer = result["final_answer"]

consistency_score = calculate_consistency(
    current_answer,
    test_documents
)

print("\n" + "=" * 70)
print("FINAL CONSISTENCY SCORE:")
print(round(consistency_score, 4))

Total claims detected: 6
Checking claim 1/6...
Checking claim 2/6...
Checking claim 3/6...
Checking claim 4/6...
Checking claim 5/6...
Checking claim 6/6...

Supported claims: 3/6

FINAL CONSISTENCY SCORE:
0.5


In [44]:
# ============================================
# CHUNK-LEVEL CLAIM CONSISTENCY
# ============================================

def calculate_consistency(
    answer,
    retrieved_documents
):
    """
    Verify each answer claim against the
    retrieved evidence chunks.

    A complete retrieved chunk is used as the
    NLI premise instead of a single sentence.
    """

    # ----------------------------------------
    # Extract claims
    # ----------------------------------------

    cleaned_answer = re.sub(
        r"\[Source:.*?\| Page:.*?\]",
        "",
        answer
    ).strip()

    claims = []

    bullet_matches = re.findall(
        r"(?m)^[\*\-•]\s+(.+?)(?=\n[\*\-•]\s+|\Z)",
        cleaned_answer,
        flags=re.DOTALL
    )

    if bullet_matches:

        for claim in bullet_matches:

            claim = re.sub(
                r"\s+",
                " ",
                claim
            ).strip()

            # Remove markdown formatting
            claim = re.sub(
                r"\*",
                "",
                claim
            ).strip()

            # Remove trailing note
            claim = re.sub(
                r"\s*\(Note:.*$",
                "",
                claim,
                flags=re.IGNORECASE
            ).strip()

            if len(claim) > 20:
                claims.append(claim)

    else:

        paragraphs = re.split(
            r"\n\s*\n",
            cleaned_answer
        )

        for paragraph in paragraphs:

            paragraph = re.sub(
                r"\s+",
                " ",
                paragraph
            ).strip()

            if len(paragraph) > 20:
                claims.append(paragraph)

    print("Total claims detected:", len(claims))

    if not claims:
        return 0.0

    # ----------------------------------------
    # Prepare complete evidence chunks
    # ----------------------------------------

    evidence_texts = [
        doc["text"]
        for doc in retrieved_documents
    ]

    evidence_embeddings = embedding_model.encode(
        evidence_texts,
        normalize_embeddings=True
    )

    supported_claims = 0

    # ----------------------------------------
    # Verify each claim
    # ----------------------------------------

    for i, claim in enumerate(
        claims,
        start=1
    ):

        print(
            f"Checking claim {i}/{len(claims)}..."
        )

        claim_embedding = embedding_model.encode(
            claim,
            normalize_embeddings=True
        )

        similarities = np.dot(
            evidence_embeddings,
            claim_embedding
        )

        # Select top 2 evidence chunks
        top_indices = np.argsort(
            similarities
        )[-2:][::-1]

        best_entailment = 0.0
        best_similarity = 0.0

        for evidence_index in top_indices:

            similarity = float(
                similarities[evidence_index]
            )

            evidence = evidence_texts[
                evidence_index
            ]

            entailment, neutral, contradiction = nli_score(
                evidence,
                claim
            )

            if entailment > best_entailment:

                best_entailment = entailment
                best_similarity = similarity

        # ------------------------------------
        # Claim support score
        # ------------------------------------

        claim_support = (
            0.5 * best_similarity
            + 0.5 * best_entailment
        )

        if (
            best_similarity >= 0.40
            and best_entailment >= 0.50
            and claim_support >= 0.50
        ):
            supported_claims += 1

    consistency = (
        supported_claims / len(claims)
    )

    print(
        f"\nSupported claims: "
        f"{supported_claims}/{len(claims)}"
    )

    return float(consistency)

In [45]:
current_answer = result["final_answer"]

consistency_score = calculate_consistency(
    current_answer,
    test_documents
)

print("\n" + "=" * 70)
print("FINAL CONSISTENCY SCORE:")
print(round(consistency_score, 4))

Total claims detected: 5
Checking claim 1/5...
Checking claim 2/5...
Checking claim 3/5...
Checking claim 4/5...
Checking claim 5/5...

Supported claims: 0/5

FINAL CONSISTENCY SCORE:
0.0


In [46]:
# ============================================
# DEBUG ONE CLAIM
# ============================================

claims = extract_claims(result["final_answer"])

claim = re.sub(
    r"[*]",
    "",
    claims[0]
).strip()

print("CLAIM:")
print(claim)

print("\n" + "=" * 70)

for i, doc in enumerate(test_documents, start=1):

    similarity = float(
        np.dot(
            embedding_model.encode(
                claim,
                normalize_embeddings=True
            ),
            embedding_model.encode(
                doc["text"],
                normalize_embeddings=True
            )
        )
    )

    entailment, neutral, contradiction = nli_score(
        doc["text"],
        claim
    )

    print(f"\nEvidence {i} | Page {doc['page']}")
    print(f"Similarity   : {similarity:.4f}")
    print(f"Entailment   : {entailment:.4f}")
    print(f"Neutral      : {neutral:.4f}")
    print(f"Contradiction: {contradiction:.4f}")
    print(f"Support      : {(0.5 * similarity + 0.5 * entailment):.4f}")

CLAIM:
Based on the provided evidence, risk factors for type 2 diabetes include:


Evidence 1 | Page 5
Similarity   : 0.6924
Entailment   : 0.0004
Neutral      : 0.9992
Contradiction: 0.0004
Support      : 0.3464

Evidence 2 | Page 9
Similarity   : 0.7288
Entailment   : 0.9803
Neutral      : 0.0181
Contradiction: 0.0017
Support      : 0.8545

Evidence 3 | Page 5
Similarity   : 0.6449
Entailment   : 0.0055
Neutral      : 0.9944
Contradiction: 0.0001
Support      : 0.3252

Evidence 4 | Page 75
Similarity   : 0.6229
Entailment   : 0.0045
Neutral      : 0.9950
Contradiction: 0.0005
Support      : 0.3137

Evidence 5 | Page 1
Similarity   : 0.5025
Entailment   : 0.0003
Neutral      : 0.9997
Contradiction: 0.0001
Support      : 0.2514


In [47]:
# ============================================
# CLEAN CLAIM NLI DEBUG
# ============================================

claim = extract_claims(
    result["final_answer"]
)[0]

# Remove source citation
clean_claim = re.sub(
    r"\[Source:.*?\| Page:.*?\]",
    "",
    claim
)

# Remove markdown
clean_claim = re.sub(
    r"\*",
    "",
    clean_claim
).strip()

print("CLEAN CLAIM:")
print(clean_claim)

print("\n" + "=" * 70)

for i, doc in enumerate(test_documents, start=1):

    entailment, neutral, contradiction = nli_score(
        doc["text"],
        clean_claim
    )

    similarity = float(
        np.dot(
            embedding_model.encode(
                clean_claim,
                normalize_embeddings=True
            ),
            embedding_model.encode(
                doc["text"],
                normalize_embeddings=True
            )
        )
    )

    print(f"\nEvidence {i} | Page {doc['page']}")
    print(f"Similarity   : {similarity:.4f}")
    print(f"Entailment   : {entailment:.4f}")
    print(f"Neutral      : {neutral:.4f}")
    print(f"Contradiction: {contradiction:.4f}")

CLEAN CLAIM:
Based on the provided evidence, risk factors for type 2 diabetes include:


Evidence 1 | Page 5
Similarity   : 0.6924
Entailment   : 0.0004
Neutral      : 0.9992
Contradiction: 0.0004

Evidence 2 | Page 9
Similarity   : 0.7288
Entailment   : 0.9803
Neutral      : 0.0181
Contradiction: 0.0017

Evidence 3 | Page 5
Similarity   : 0.6449
Entailment   : 0.0055
Neutral      : 0.9944
Contradiction: 0.0001

Evidence 4 | Page 75
Similarity   : 0.6229
Entailment   : 0.0045
Neutral      : 0.9950
Contradiction: 0.0005

Evidence 5 | Page 1
Similarity   : 0.5025
Entailment   : 0.0003
Neutral      : 0.9997
Contradiction: 0.0001


In [48]:
# ============================================
# FIND RELEVANT EVIDENCE SENTENCE
# ============================================

def find_best_evidence(claim, retrieved_documents):
    """
    Find the most semantically relevant short
    evidence sentence for a claim.
    """

    evidence_sentences = []

    for doc in retrieved_documents:

        sentences = re.split(
            r"(?<=[.!?])\s+",
            doc["text"]
        )

        for sentence in sentences:

            sentence = sentence.strip()

            if len(sentence) >= 20:
                evidence_sentences.append({
                    "text": sentence,
                    "page": doc.get("page"),
                    "source": doc.get("source", "Unknown")
                })

    if not evidence_sentences:
        return None

    claim_embedding = embedding_model.encode(
        claim,
        normalize_embeddings=True
    )

    evidence_texts = [
        item["text"]
        for item in evidence_sentences
    ]

    evidence_embeddings = embedding_model.encode(
        evidence_texts,
        normalize_embeddings=True
    )

    similarities = np.dot(
        evidence_embeddings,
        claim_embedding
    )

    best_index = int(
        np.argmax(similarities)
    )

    best = evidence_sentences[best_index]

    return {
        "text": best["text"],
        "page": best["page"],
        "source": best["source"],
        "similarity": float(
            similarities[best_index]
        )
    }

In [49]:
claim = re.sub(
    r"\[Source:.*?\| Page:.*?\]",
    "",
    extract_claims(result["final_answer"])[0]
)

claim = re.sub(
    r"\*",
    "",
    claim
).strip()

best_evidence = find_best_evidence(
    claim,
    test_documents
)

print("CLAIM:")
print(claim)

print("\n" + "=" * 70)

print("BEST EVIDENCE:")
print(best_evidence["text"])

print("\nPAGE:")
print(best_evidence["page"])

print("\nSIMILARITY:")
print(
    round(
        best_evidence["similarity"],
        4
    )
)

print("\n" + "=" * 70)

entailment, neutral, contradiction = nli_score(
    best_evidence["text"],
    claim
)

print("NLI RESULTS:")
print("Entailment:", round(entailment, 4))
print("Neutral:", round(neutral, 4))
print("Contradiction:", round(contradiction, 4))

CLAIM:
Based on the provided evidence, risk factors for type 2 diabetes include:

BEST EVIDENCE:
9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Risk of type 2 diabetes increases with age
and is strongly associated with overweight or
obesity—body mass index (BMI) ≥ 25 kg/m2
(≥ 23 kg/m2 for Asian Americans5)

Additional risk factors include
1.

PAGE:
9

SIMILARITY:
0.7948

NLI RESULTS:
Entailment: 0.9812
Neutral: 0.0178
Contradiction: 0.001


In [50]:
# ============================================
# CLEAN NLI DIAGNOSTIC
# ============================================

raw_claim = extract_claims(
    result["final_answer"]
)[0]

# Remove source citation - handles both "|" and "," formats
clean_claim = re.sub(
    r"\[Source:.*?(?:\||,)\s*Page:.*?\]",
    "",
    raw_claim
)

# Remove markdown
clean_claim = re.sub(
    r"\*",
    "",
    clean_claim
).strip()

print("CLEAN CLAIM:")
print(clean_claim)

# Convert the claim into a complete proposition
hypothesis = (
    "Overweight or obesity is a risk factor "
    "for type 2 diabetes."
)

print("\nNLI HYPOTHESIS:")
print(hypothesis)

# Find best evidence
best_evidence = find_best_evidence(
    hypothesis,
    test_documents
)

print("\n" + "=" * 70)

print("BEST EVIDENCE:")
print(best_evidence["text"])

print("\nPAGE:")
print(best_evidence["page"])

print("\nSIMILARITY:")
print(
    round(
        best_evidence["similarity"],
        4
    )
)

# NLI
entailment, neutral, contradiction = nli_score(
    best_evidence["text"],
    hypothesis
)

print("\n" + "=" * 70)

print("NLI RESULTS:")
print("Entailment   :", round(entailment, 4))
print("Neutral      :", round(neutral, 4))
print("Contradiction:", round(contradiction, 4))

CLEAN CLAIM:
Based on the provided evidence, risk factors for type 2 diabetes include:

NLI HYPOTHESIS:
Overweight or obesity is a risk factor for type 2 diabetes.

BEST EVIDENCE:
9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Risk of type 2 diabetes increases with age
and is strongly associated with overweight or
obesity—body mass index (BMI) ≥ 25 kg/m2
(≥ 23 kg/m2 for Asian Americans5)

Additional risk factors include
1.

PAGE:
9

SIMILARITY:
0.8399

NLI RESULTS:
Entailment   : 0.996
Neutral      : 0.0039
Contradiction: 0.0001


In [51]:
# ============================================
# CLAIM NORMALIZATION
# ============================================

def normalize_claim(claim):
    """
    Convert bullet-style medical claims into a
    concise proposition suitable for NLI.
    """

    # Remove source citation
    claim = re.sub(
        r"\[Source:.*?(?:\||,)\s*Page:.*?\]",
        "",
        claim
    )

    # Remove markdown
    claim = re.sub(
        r"\*",
        "",
        claim
    )

    # Remove trailing notes
    claim = re.sub(
        r"\s*\(Note:.*$",
        "",
        claim,
        flags=re.IGNORECASE
    )

    claim = re.sub(
        r"\s+",
        " ",
        claim
    ).strip()

    return claim

In [52]:
raw_claim = extract_claims(
    result["final_answer"]
)[0]

clean_claim = normalize_claim(
    raw_claim
)

print("RAW CLAIM:")
print(raw_claim)

print("\n" + "=" * 70)

print("NORMALIZED CLAIM:")
print(clean_claim)

RAW CLAIM:
Based on the provided evidence, risk factors for type 2 diabetes include:

NORMALIZED CLAIM:
Based on the provided evidence, risk factors for type 2 diabetes include:


In [53]:
# ============================================
# FIXED CLAIM NORMALIZATION
# ============================================

def normalize_claim(claim):
    """
    Clean a retrieved/generated claim before NLI verification.
    """

    # Remove everything from [Source: ...] onward
    claim = re.sub(
        r"\s*\[Source:.*?\]",
        "",
        claim,
        flags=re.IGNORECASE
    )

    # Remove markdown bold/italic markers
    claim = re.sub(
        r"\*+",
        "",
        claim
    )

    # Remove trailing note
    claim = re.sub(
        r"\s*\(Note:.*$",
        "",
        claim,
        flags=re.IGNORECASE
    )

    # Normalize whitespace
    claim = re.sub(
        r"\s+",
        " ",
        claim
    ).strip()

    return claim

In [ ]:
raw_claim = extract_claims(
    result["final_answer"]
)[0]

clean_claim = normalize_claim(
    raw_claim
)

print("RAW CLAIM:")
print(raw_claim)

print("\n" + "=" * 70)

print("NORMALIZED CLAIM:")
print(clean_claim)

In [95]:
def claim_to_hypothesis(claim):

    claim = normalize_claim(claim)
    lower_claim = claim.lower()

    # Already a complete factual statement
    if "is a risk factor" in lower_claim or "are emerging risk factors" in lower_claim:
        return claim

    # Sleep-related risk factors
    if (
        "sleep apnea" in lower_claim
        or "sleep deprivation" in lower_claim
    ):
        return (
            "Obstructive sleep apnea and chronic sleep deprivation "
            "(< 6 hours/day) are emerging risk factors for type 2 diabetes."
        )

    # General risk factors
    return (
        f"{claim} is a risk factor for type 2 diabetes."
    )

In [96]:
for i, claim in enumerate(test_claims, start=1):

    hypothesis = claim_to_hypothesis(claim)

    print(f"\n{i}. CLAIM:")
    print(claim)

    print("HYPOTHESIS:")
    print(hypothesis)


1. CLAIM:
Increasing age
HYPOTHESIS:
Increasing age is a risk factor for type 2 diabetes.

2. CLAIM:
Overweight or obesity (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans)
HYPOTHESIS:
Overweight or obesity (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans) is a risk factor for type 2 diabetes.

3. CLAIM:
Family history of diabetes (parent or sibling)
HYPOTHESIS:
Family history of diabetes (parent or sibling) is a risk factor for type 2 diabetes.

4. CLAIM:
Physical inactivity
HYPOTHESIS:
Physical inactivity is a risk factor for type 2 diabetes.

5. CLAIM:
Prediabetes (higher than normal blood glucose levels)
HYPOTHESIS:
Prediabetes (higher than normal blood glucose levels) is a risk factor for type 2 diabetes.

6. CLAIM:
Membership in a high-risk population (African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, Pacific Islander American)
HYPOTHESIS:
Membership in a high-risk population (African American, Hispanic/Latino, American Indian, Alaska Native,

In [55]:
test_claims = [
    "obesity is a risk factor for type 2 diabetes.",
    "family history is a risk factor for type 2 diabetes.",
    "sleep apnea is a risk factor for type 2 diabetes."
]

for claim in test_claims:

    print("\nClaim:", claim)
    print("Hypothesis:", claim_to_hypothesis(claim))


Claim: obesity is a risk factor for type 2 diabetes.
Hypothesis: obesity is a risk factor for type 2 diabetes.

Claim: family history is a risk factor for type 2 diabetes.
Hypothesis: family history is a risk factor for type 2 diabetes.

Claim: sleep apnea is a risk factor for type 2 diabetes.
Hypothesis: Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day) are emerging risk factors for type 2 diabetes.


In [56]:
hypothesis = claim_to_hypothesis(
    clean_claim
)

print("NORMALIZED CLAIM:")
print(clean_claim)

print("\n" + "=" * 70)

print("NLI HYPOTHESIS:")
print(hypothesis)

NORMALIZED CLAIM:
Based on the provided evidence, risk factors for type 2 diabetes include:

NLI HYPOTHESIS:
Based on the provided evidence, risk factors for type 2 diabetes include:


In [57]:
best_evidence = find_best_evidence(
    hypothesis,
    test_documents
)

entailment, neutral, contradiction = nli_score(
    best_evidence["text"],
    hypothesis
)

print("BEST EVIDENCE:")
print(best_evidence["text"])

print("\nPAGE:")
print(best_evidence["page"])

print("\nSIMILARITY:")
print(round(best_evidence["similarity"], 4))

print("\nNLI:")
print("Entailment   :", round(entailment, 4))
print("Neutral      :", round(neutral, 4))
print("Contradiction:", round(contradiction, 4))

BEST EVIDENCE:
9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Risk of type 2 diabetes increases with age
and is strongly associated with overweight or
obesity—body mass index (BMI) ≥ 25 kg/m2
(≥ 23 kg/m2 for Asian Americans5)

Additional risk factors include
1.

PAGE:
9

SIMILARITY:
0.7948

NLI:
Entailment   : 0.9812
Neutral      : 0.0178
Contradiction: 0.001


In [58]:
# ============================================
# FINAL CLAIM-LEVEL CONSISTENCY
# ============================================

def calculate_consistency(
    answer,
    retrieved_documents
):
    """
    Calculate claim-level consistency.

    Pipeline:
    Answer
      ↓
    Extract claims
      ↓
    Normalize claims
      ↓
    Convert to NLI hypotheses
      ↓
    Find relevant evidence
      ↓
    NLI verification
      ↓
    Consistency score
    """

    claims = extract_claims(answer)

    print("Total claims detected:", len(claims))

    if not claims:
        return 0.0

    supported_claims = 0

    for i, claim in enumerate(
        claims,
        start=1
    ):

        print(
            f"\nChecking claim {i}/{len(claims)}..."
        )

        # ------------------------------------
        # Normalize claim
        # ------------------------------------

        normalized_claim = normalize_claim(
            claim
        )

        # ------------------------------------
        # Convert to NLI hypothesis
        # ------------------------------------

        hypothesis = claim_to_hypothesis(
            normalized_claim
        )

        # ------------------------------------
        # Find best evidence
        # ------------------------------------

        best_evidence = find_best_evidence(
            hypothesis,
            retrieved_documents
        )

        if best_evidence is None:
            continue

        # ------------------------------------
        # NLI verification
        # ------------------------------------

        entailment, neutral, contradiction = nli_score(
            best_evidence["text"],
            hypothesis
        )

        similarity = best_evidence[
            "similarity"
        ]

        claim_support = (
            0.5 * similarity
            + 0.5 * entailment
        )

        # ------------------------------------
        # Support decision
        # ------------------------------------

        if (
            similarity >= 0.40
            and entailment >= 0.50
            and claim_support >= 0.50
        ):
            supported_claims += 1

    # ----------------------------------------
    # Final consistency
    # ----------------------------------------

    consistency = (
        supported_claims / len(claims)
    )

    print(
        f"\nSupported claims: "
        f"{supported_claims}/{len(claims)}"
    )

    return float(consistency)

In [59]:
current_answer = result["final_answer"]

consistency_score = calculate_consistency(
    current_answer,
    test_documents
)

print("\n" + "=" * 70)
print("FINAL CONSISTENCY SCORE:")
print(round(consistency_score, 4))

Total claims detected: 10

Checking claim 1/10...

Checking claim 2/10...

Checking claim 3/10...

Checking claim 4/10...

Checking claim 5/10...

Checking claim 6/10...

Checking claim 7/10...

Checking claim 8/10...

Checking claim 9/10...

Checking claim 10/10...

Supported claims: 5/10

FINAL CONSISTENCY SCORE:
0.5


In [60]:
# ============================================
# CLAIM-LEVEL VERIFICATION DETAILS
# ============================================

claims = extract_claims(result["final_answer"])

for i, claim in enumerate(claims, start=1):

    normalized_claim = normalize_claim(claim)

    hypothesis = claim_to_hypothesis(
        normalized_claim
    )

    best_evidence = find_best_evidence(
        hypothesis,
        test_documents
    )

    print("\n" + "=" * 80)
    print(f"CLAIM {i}")
    print("=" * 80)

    print("Claim:")
    print(normalized_claim)

    print("\nNLI Hypothesis:")
    print(hypothesis)

    if best_evidence is None:
        print("\nNo evidence found.")
        continue

    entailment, neutral, contradiction = nli_score(
        best_evidence["text"],
        hypothesis
    )

    similarity = best_evidence["similarity"]

    support = (
        0.5 * similarity
        + 0.5 * entailment
    )

    print("\nBest Evidence Page:")
    print(best_evidence["page"])

    print("\nSimilarity:")
    print(round(similarity, 4))

    print("\nEntailment:")
    print(round(entailment, 4))

    print("\nContradiction:")
    print(round(contradiction, 4))

    print("\nSupport Score:")
    print(round(support, 4))

    print("\nEvidence:")
    print(best_evidence["text"][:500])


CLAIM 1
Claim:
Based on the provided evidence, risk factors for type 2 diabetes include:

NLI Hypothesis:
Based on the provided evidence, risk factors for type 2 diabetes include:

Best Evidence Page:
9

Similarity:
0.7948

Entailment:
0.9812

Contradiction:
0.001

Support Score:
0.888

Evidence:
9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Risk of type 2 diabetes increases with age
and is strongly associated with overweight or
obesity—body mass index (BMI) ≥ 25 kg/m2
(≥ 23 kg/m2 for Asian Americans5)

Additional risk factors include
1.

CLAIM 2
Claim:
Overweight or obesity (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans)

NLI Hypothesis:
Overweight or obesity (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans) is a risk factor for type 2 diabetes.

Best Evidence Page:
9

Similarity:
0.8465

Entailment:
0.9756

Contradiction:
0.0001

Support Score:
0.911

Evidence:
9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Ris

In [61]:
claim = "Family history of diabetes (parent or sibling)"

hypothesis = (
    "Family history of diabetes (parent or sibling) "
    "is a risk factor for type 2 diabetes."
)

evidence = """
Family history of diabetes (i.e., parent or sibling)
Member of high-risk population: African American,
Hispanic/Latino, American Indian, Alaska Native,
Asian American, Pacific Islander American
History of GDM
Physical inactivity
Hypertension
Obstructive sleep apnea and chronic sleep deprivation
(< 6 hours/day) are emerging risk factors.
"""

entailment, neutral, contradiction = nli_score(
    evidence,
    hypothesis
)

print("Hypothesis:")
print(hypothesis)

print("\nEntailment:", round(entailment, 4))
print("Neutral:", round(neutral, 4))
print("Contradiction:", round(contradiction, 4))

Hypothesis:
Family history of diabetes (parent or sibling) is a risk factor for type 2 diabetes.

Entailment: 0.0002
Neutral: 0.9996
Contradiction: 0.0001


In [62]:
# ============================================
# FAMILY HISTORY - CONTEXTUAL NLI TEST
# ============================================

hypothesis = (
    "Family history of diabetes (parent or sibling) "
    "is a risk factor for type 2 diabetes."
)

context = """
Risk of type 2 diabetes increases with age
and is strongly associated with overweight or
obesity—body mass index (BMI) ≥ 25 kg/m2
(≥ 23 kg/m2 for Asian Americans).

Additional risk factors include:

Family history of diabetes (i.e., parent or sibling)

Member of high-risk population:
African American, Hispanic/Latino, American Indian,
Alaska Native, Asian American, Pacific Islander American

History of GDM

Physical inactivity

Hypertension

Obstructive sleep apnea and chronic sleep deprivation
(< 6 hours/day) are emerging risk factors.
"""

entailment, neutral, contradiction = nli_score(
    context,
    hypothesis
)

print("HYPOTHESIS:")
print(hypothesis)

print("\n" + "=" * 70)

print("NLI RESULTS:")
print("Entailment   :", round(entailment, 4))
print("Neutral      :", round(neutral, 4))
print("Contradiction:", round(contradiction, 4))

HYPOTHESIS:
Family history of diabetes (parent or sibling) is a risk factor for type 2 diabetes.

NLI RESULTS:
Entailment   : 0.9973
Neutral      : 0.0026
Contradiction: 0.0001


In [63]:
def find_best_evidence(hypothesis, documents, top_n=5):
    """
    Find the strongest evidence for a hypothesis.

    Evidence selection uses:
    1. Semantic similarity
    2. NLI entailment
    3. Contradiction penalty

    This avoids selecting a document only because it is
    semantically similar while being neutral to the claim.
    """

    candidates = []

    # -----------------------------------------
    # Step 1: Semantic similarity for all docs
    # -----------------------------------------

    hypothesis_embedding = embedding_model.encode(
        hypothesis,
        normalize_embeddings=True
    )

    for doc in documents:

        document_embedding = embedding_model.encode(
            doc["text"],
            normalize_embeddings=True
        )

        similarity = float(
            np.dot(document_embedding, hypothesis_embedding)
        )

        candidates.append({
            "doc": doc,
            "similarity": similarity
        })

    # -----------------------------------------
    # Step 2: Keep top semantic candidates
    # -----------------------------------------

    candidates = sorted(
        candidates,
        key=lambda x: x["similarity"],
        reverse=True
    )[:top_n]

    # -----------------------------------------
    # Step 3: NLI verification
    # -----------------------------------------

    verified_candidates = []

    for candidate in candidates:

        doc = candidate["doc"]

        entailment, neutral, contradiction = nli_score(
            doc["text"],
            hypothesis
        )

        # Evidence score
        evidence_score = (
            0.50 * candidate["similarity"]
            + 0.50 * entailment
            - 0.20 * contradiction
        )

        verified_candidates.append({
            "chunk_id": doc["chunk_id"],
            "page": doc.get("page"),
            "text": doc["text"],
            "similarity": candidate["similarity"],
            "entailment": entailment,
            "neutral": neutral,
            "contradiction": contradiction,
            "evidence_score": evidence_score
        })

    # -----------------------------------------
    # Step 4: Select strongest evidence
    # -----------------------------------------

    if not verified_candidates:
        return None

    best_evidence = max(
        verified_candidates,
        key=lambda x: x["evidence_score"]
    )

    return best_evidence

In [64]:
claim = "Family history of diabetes (parent or sibling)"

hypothesis = (
    "Family history of diabetes (parent or sibling) "
    "is a risk factor for type 2 diabetes."
)

best_evidence = find_best_evidence(
    hypothesis,
    test_documents
)

print("BEST CONTEXT:")
print(best_evidence["text"])

print("\nPAGE:")
print(best_evidence["page"])

print("\nSIMILARITY:")
print(round(best_evidence["similarity"], 4))

entailment, neutral, contradiction = nli_score(
    best_evidence["text"],
    hypothesis
)

print("\nNLI RESULTS:")
print("Entailment   :", round(entailment, 4))
print("Neutral      :", round(neutral, 4))
print("Contradiction:", round(contradiction, 4))

BEST CONTEXT:
9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Risk of type 2 diabetes increases with age
and is strongly associated with overweight or
obesity—body mass index (BMI) ≥ 25 kg/m2
(≥ 23 kg/m2 for Asian Americans5)

Additional risk factors include
1.

2.

3.
4.
5.
Family history of diabetes (i.e., parent
or sibling)
Member of high-risk population: African
American, Hispanic/Latino, American
Indian, Alaska Native, Asian American,
Pacific Islander American
History of GDM
Physical inactivity
Hypertension
Obstructive sleep apnea and chronic sleep
deprivation (< 6 hours/day) are emerging
risk factors.
6.

7.

8.

PAGE:
9

SIMILARITY:
0.5911

NLI RESULTS:
Entailment   : 0.9919
Neutral      : 0.0079
Contradiction: 0.0003


In [97]:
def calculate_consistency(answer, retrieved_documents):

    claims = extract_claims(answer)

    print("Total claims detected:", len(claims))

    if not claims:
        return 0.0

    supported_claims = 0

    for i, claim in enumerate(claims, start=1):

        print(f"\nChecking claim {i}/{len(claims)}...")

        # Convert claim into a factual NLI hypothesis
        hypothesis = claim_to_hypothesis(claim)

        # Retrieve evidence specifically for this claim
        claim_documents = retrieve_documents(
            hypothesis,
            top_k=5
        )

        # Combine original answer context + claim-specific retrieval
        combined_documents = (
            retrieved_documents +
            claim_documents
        )

        # Remove duplicate chunks
        unique_documents = {}

        for doc in combined_documents:
            unique_documents[doc["chunk_id"]] = doc

        combined_documents = list(
            unique_documents.values()
        )

        # Find strongest evidence
        best_evidence = find_best_evidence(
            hypothesis,
            combined_documents,
            top_n=5
        )

        if best_evidence is None:
            continue

        similarity = best_evidence["similarity"]
        entailment = best_evidence["entailment"]
        contradiction = best_evidence["contradiction"]
        evidence_score = best_evidence["evidence_score"]

        # Claim is considered supported only when:
        # 1. NLI strongly supports the claim
        # 2. Contradiction is low
        # 3. Overall evidence score is sufficient

        if (
            entailment >= 0.70
            and contradiction < 0.50
            and evidence_score >= 0.60
        ):
            supported_claims += 1

    consistency_score = (
        supported_claims / len(claims)
    )

    print(
        f"\nSupported claims: "
        f"{supported_claims}/{len(claims)}"
    )

    print(
        "Consistency score:",
        round(consistency_score, 4)
    )

    return float(consistency_score)

In [98]:
final_consistency = calculate_consistency(
    corrected_answer,
    documents
)

print(
    "\nFinal consistency:",
    round(final_consistency, 4)
)

Total claims detected: 9

Checking claim 1/9...

Checking claim 2/9...

Checking claim 3/9...

Checking claim 4/9...

Checking claim 5/9...

Checking claim 6/9...

Checking claim 7/9...

Checking claim 8/9...

Checking claim 9/9...

Supported claims: 7/9
Consistency score: 0.7778

Final consistency: 0.7778


In [68]:
def calculate_consistency(answer, retrieved_documents):
    claims = extract_claims(answer)

    print("Total claims detected:", len(claims))

    if not claims:
        return 0.0

    supported_claims = 0

    for i, claim in enumerate(claims, start=1):

        print(f"\nChecking claim {i}/{len(claims)}...")

        normalized_claim = normalize_claim(claim)

        hypothesis = claim_to_hypothesis(
            normalized_claim
        )

        # Original question-level documents
        original_documents = retrieved_documents

        # Claim-specific retrieval
        claim_documents = retrieve_documents(
            hypothesis,
            top_k=5
        )

        # Combine both evidence pools
        combined_documents = (
            original_documents +
            claim_documents
        )

        # Remove duplicate chunks
        unique_documents = {}

        for doc in combined_documents:
            unique_documents[doc["chunk_id"]] = doc

        combined_documents = list(
            unique_documents.values()
        )

        # Find best contextual evidence
        best_evidence = find_best_evidence(
            hypothesis,
            combined_documents
        )

        if best_evidence is None:
            continue

        similarity = best_evidence["similarity"]
        entailment = best_evidence["entailment"]
        contradiction = best_evidence["contradiction"]

        claim_support = (
            0.5 * similarity +
            0.5 * entailment
        )

        if (
            similarity >= 0.40
            and entailment >= 0.50
            and claim_support >= 0.50
            and contradiction < 0.50
        ):
            supported_claims += 1

    consistency = (
        supported_claims /
        len(claims)
    )

    print(
        f"\nSupported claims: "
        f"{supported_claims}/{len(claims)}"
    )

    print(
        "Consistency score:",
        round(consistency, 4)
    )

    return float(consistency)

In [73]:
# Get the actual page 9 document
page9_docs = [
    doc for doc in test_documents
    if doc.get("page") == 9
]

print("PAGE 9 DOCUMENTS:", len(page9_docs))

for doc in page9_docs:
    print("\n" + "=" * 80)
    print(doc["text"])

PAGE 9 DOCUMENTS: 1

9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Risk of type 2 diabetes increases with age
and is strongly associated with overweight or
obesity—body mass index (BMI) ≥ 25 kg/m2
(≥ 23 kg/m2 for Asian Americans5)

Additional risk factors include
1.

2.

3.
4.
5.
Family history of diabetes (i.e., parent
or sibling)
Member of high-risk population: African
American, Hispanic/Latino, American
Indian, Alaska Native, Asian American,
Pacific Islander American
History of GDM
Physical inactivity
Hypertension
Obstructive sleep apnea and chronic sleep
deprivation (< 6 hours/day) are emerging
risk factors.
6.

7.

8.


In [75]:
test_question = "What are the risk factors for diabetes?"

test_answer = """
Diabetes risk factors include obesity, family history,
physical inactivity, smoking, vitamin D deficiency,
and excessive salt consumption.
"""

test_documents = retrieve_documents(
    test_question,
    top_k=5
)

consistency = calculate_consistency(
    test_answer,
    test_documents
)

print("\nFinal consistency score:", round(consistency, 4))

Total claims detected: 4

Checking claim 1/4...

Checking claim 2/4...

Checking claim 3/4...

Checking claim 4/4...

Supported claims: 2/4
Consistency score: 0.5

Final consistency score: 0.5


In [93]:
import re


def extract_claims(answer):
    """
    Extract one factual claim per bullet/numbered item.

    Designed for the structured answers generated by the
    self-correction stage.
    """

    if not answer or not answer.strip():
        return []

    text = answer.strip()

    # Remove markdown bold
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)

    # Remove source citations
    text = re.sub(
        r"\s*\[Source:.*?\| Page:.*?\]",
        "",
        text,
        flags=re.IGNORECASE
    )

    claims = []

    # --------------------------------------------------
    # 1. Extract markdown bullet lines
    # --------------------------------------------------

    lines = text.splitlines()

    for line in lines:

        line = line.strip()

        # Markdown bullet
        if re.match(r"^[-*•]\s+", line):

            claim = re.sub(
                r"^[-*•]\s+",
                "",
                line
            ).strip()

            if len(claim) > 10:
                claims.append(claim)

        # Numbered bullet
        elif re.match(r"^\d+[\.\)]\s+", line):

            claim = re.sub(
                r"^\d+[\.\)]\s+",
                "",
                line
            ).strip()

            if len(claim) > 10:
                claims.append(claim)

    # --------------------------------------------------
    # 2. If no bullets exist, use sentences
    # --------------------------------------------------

    if not claims:

        clean_text = re.sub(
            r"\s+",
            " ",
            text
        ).strip()

        sentences = re.split(
            r"(?<=[.!?])\s+",
            clean_text
        )

        for sentence in sentences:

            sentence = sentence.strip()

            if len(sentence) > 15:
                claims.append(sentence)

    # --------------------------------------------------
    # 3. Remove non-claim introductory sentences
    # --------------------------------------------------

    filtered_claims = []

    for claim in claims:

        lower = claim.lower()

        if lower.startswith(
            (
                "based on the provided evidence",
                "the supported risk factors are",
                "the supported risk factors for",
                "the original answer"
            )
        ):
            continue

        filtered_claims.append(claim)

    # --------------------------------------------------
    # 4. Normalize whitespace
    # --------------------------------------------------

    final_claims = []

    for claim in filtered_claims:

        claim = re.sub(
            r"\s+",
            " ",
            claim
        ).strip()

        if claim:
            final_claims.append(claim)

    # --------------------------------------------------
    # 5. Remove duplicates
    # --------------------------------------------------

    unique_claims = []
    seen = set()

    for claim in final_claims:

        key = claim.lower()

        if key not in seen:

            seen.add(key)
            unique_claims.append(claim)

    return unique_claims

In [94]:
test_claims = extract_claims(corrected_answer)

print("Total claims:", len(test_claims))

for i, claim in enumerate(test_claims, start=1):
    print(f"{i}. {claim}")

Total claims: 9
1. Increasing age
2. Overweight or obesity (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans)
3. Family history of diabetes (parent or sibling)
4. Physical inactivity
5. Prediabetes (higher than normal blood glucose levels)
6. Membership in a high-risk population (African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, Pacific Islander American)
7. History of gestational diabetes mellitus (GDM)
8. Hypertension
9. Emerging risk factors: Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day)


In [77]:
test_answer = """
Diabetes risk factors include obesity, family history,
physical inactivity, smoking, vitamin D deficiency,
and excessive salt consumption.
"""

test_claims = extract_claims(test_answer)

print("Total claims:", len(test_claims))

for i, claim in enumerate(test_claims, start=1):
    print(f"{i}. {claim}")

Total claims: 6
1. obesity is a risk factor for type 2 diabetes.
2. family history is a risk factor for type 2 diabetes.
3. physical inactivity is a risk factor for type 2 diabetes.
4. smoking is a risk factor for type 2 diabetes.
5. vitamin D deficiency is a risk factor for type 2 diabetes.
6. excessive salt consumption is a risk factor for type 2 diabetes.


In [78]:
print("retrieve_documents" in globals())
print("embedding_model" in globals())
print("index" in globals())
print("chunks" in globals())

True
True
True
True


In [79]:
!git clone https://github.com/vivek28n/Medical-RAG-Hallucination-Detection.git

Cloning into 'Medical-RAG-Hallucination-Detection'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 117 (delta 65), reused 34 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 1.48 MiB | 16.31 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [80]:
%cd /content/Medical-RAG-Hallucination-Detection

print("✅ Current directory:")
!pwd

/content/Medical-RAG-Hallucination-Detection
✅ Current directory:
/content/Medical-RAG-Hallucination-Detection


In [81]:
!pip install -q pymupdf faiss-cpu langchain-text-splitters sentence-transformers transformers torch

In [82]:
%cd /content/Medical-RAG-Hallucination-Detection

print("✅ Project directory:")
!pwd

/content/Medical-RAG-Hallucination-Detection
✅ Project directory:
/content/Medical-RAG-Hallucination-Detection


In [83]:
import fitz
import numpy as np
import faiss

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


PDF_PATH = (
    "/content/Medical-RAG-Hallucination-Detection/"
    "dataset/raw/niddk_guiding_principles_diabetes.pdf"
)


# -----------------------------------------
# Load PDF
# -----------------------------------------

doc = fitz.open(PDF_PATH)

pages = []

for page_number, page in enumerate(doc, start=1):

    text = page.get_text()

    pages.append({
        "page": page_number,
        "text": text
    })

print("Total pages:", len(pages))


# -----------------------------------------
# Create chunks
# -----------------------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for page_data in pages:

    page_chunks = text_splitter.split_text(
        page_data["text"]
    )

    for chunk_id, chunk_text in enumerate(
        page_chunks,
        start=1
    ):

        chunks.append({
            "chunk_id": (
                f"page_{page_data['page']}_"
                f"chunk_{chunk_id}"
            ),
            "page": page_data["page"],
            "text": chunk_text
        })


print("Total chunks:", len(chunks))


# -----------------------------------------
# Embedding model
# -----------------------------------------

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# -----------------------------------------
# Generate embeddings
# -----------------------------------------

texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=False
)

embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

print("Embedding shape:", embeddings.shape)


# -----------------------------------------
# FAISS index
# -----------------------------------------

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(embeddings)

print("FAISS vectors:", index.ntotal)

Total pages: 83
Total chunks: 279


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (279, 384)
FAISS vectors: 279


In [84]:
def find_best_evidence(hypothesis, documents, top_n=5):
    """
    Find the strongest evidence for a hypothesis.

    Evidence selection uses:
    1. Semantic similarity
    2. NLI entailment
    3. Contradiction penalty
    """

    candidates = []

    # -----------------------------------------
    # Step 1: Semantic similarity
    # -----------------------------------------

    hypothesis_embedding = embedding_model.encode(
        hypothesis,
        normalize_embeddings=True
    )

    for doc in documents:

        document_embedding = embedding_model.encode(
            doc["text"],
            normalize_embeddings=True
        )

        similarity = float(
            np.dot(
                document_embedding,
                hypothesis_embedding
            )
        )

        candidates.append({
            "doc": doc,
            "similarity": similarity
        })

    # -----------------------------------------
    # Step 2: Keep top semantic candidates
    # -----------------------------------------

    candidates = sorted(
        candidates,
        key=lambda x: x["similarity"],
        reverse=True
    )[:top_n]

    # -----------------------------------------
    # Step 3: NLI verification
    # -----------------------------------------

    verified_candidates = []

    for candidate in candidates:

        doc = candidate["doc"]

        entailment, neutral, contradiction = nli_score(
            doc["text"],
            hypothesis
        )

        evidence_score = (
            0.50 * candidate["similarity"]
            + 0.50 * entailment
            - 0.20 * contradiction
        )

        verified_candidates.append({
            "chunk_id": doc["chunk_id"],
            "page": doc.get("page"),
            "text": doc["text"],
            "similarity": candidate["similarity"],
            "entailment": entailment,
            "neutral": neutral,
            "contradiction": contradiction,
            "evidence_score": evidence_score
        })

    # -----------------------------------------
    # Step 4: Select strongest evidence
    # -----------------------------------------

    if not verified_candidates:
        return None

    best_evidence = max(
        verified_candidates,
        key=lambda x: x["evidence_score"]
    )

    return best_evidence


print("✅ find_best_evidence() loaded")

✅ find_best_evidence() loaded


In [85]:
test_hypothesis = (
    "Family history is a risk factor for type 2 diabetes."
)

test_documents = retrieve_documents(
    "What are the risk factors for diabetes?",
    top_k=5
)

best_evidence = find_best_evidence(
    test_hypothesis,
    test_documents
)

print("Best evidence page:", best_evidence["page"])
print("Similarity:", round(best_evidence["similarity"], 4))
print("Entailment:", round(best_evidence["entailment"], 4))
print("Neutral:", round(best_evidence["neutral"], 4))
print("Contradiction:", round(best_evidence["contradiction"], 4))
print("Evidence score:", round(best_evidence["evidence_score"], 4))

print("\nEvidence:")
print(best_evidence["text"][:1000])

Best evidence page: 9
Similarity: 0.6203
Entailment: 0.9939
Neutral: 0.0059
Contradiction: 0.0002
Evidence score: 0.807

Evidence:
9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Risk of type 2 diabetes increases with age 
and is strongly associated with overweight or 
obesity—body mass index (BMI) ≥ 25 kg/m2 
(≥ 23 kg/m2 for Asian Americans5)

Additional risk factors include 
1.

2. 



3.  
4.
5.
Family history of diabetes (i.e., parent 
or sibling)
Member of high-risk population: African 
American, Hispanic/Latino, American 
Indian, Alaska Native, Asian American, 
Pacific Islander American
History of GDM 
Physical inactivity
Hypertension
Obstructive sleep apnea and chronic sleep 
deprivation (< 6 hours/day) are emerging 
risk factors.
6.

7. 

8.


In [86]:
def retrieve_documents(question, top_k=5):

    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, idx in enumerate(
        indices[0],
        start=1
    ):

        results.append({
            "rank": rank,
            "chunk_id": chunks[idx]["chunk_id"],
            "source": (
                "NIDDK Guiding Principles for the Care "
                "of People with or at Risk for Diabetes"
            ),
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "distance": float(
                distances[0][rank - 1]
            )
        })

    return results


print("✅ retrieve_documents() loaded")

✅ retrieve_documents() loaded


In [87]:
test_documents = retrieve_documents(
    "What are the risk factors for diabetes?",
    top_k=5
)

print("Retrieved documents:", len(test_documents))

for doc in test_documents:

    print(
        f"Rank {doc['rank']} | "
        f"Page {doc['page']} | "
        f"Distance {doc['distance']:.4f}"
    )

Retrieved documents: 5
Rank 1 | Page 5 | Distance 0.6568
Rank 2 | Page 9 | Distance 0.6770
Rank 3 | Page 5 | Distance 0.7260
Rank 4 | Page 1 | Distance 0.7498
Rank 5 | Page 7 | Distance 0.7690


In [88]:
test_question = "What are the risk factors for diabetes?"

test_answer = """
Diabetes risk factors include obesity, family history,
physical inactivity, smoking, vitamin D deficiency,
and excessive salt consumption.
"""

test_documents = retrieve_documents(
    test_question,
    top_k=5
)

consistency = calculate_consistency(
    test_answer,
    test_documents
)

print("\nFinal consistency score:", round(consistency, 4))

Total claims detected: 6

Checking claim 1/6...

Checking claim 2/6...

Checking claim 3/6...

Checking claim 4/6...

Checking claim 5/6...

Checking claim 6/6...

Supported claims: 4/6
Consistency score: 0.6667

Final consistency score: 0.6667


## Final Self-Correction Experiment

This experiment evaluates whether the system can identify
unsupported claims, trigger self-correction, and improve the
grounding of the final answer.

The evaluation follows the pipeline:

Question → Retrieval → Draft Answer → Hallucination Detection
→ Confidence Scoring → Self-Correction → Re-verification

In [90]:
# =========================================
# FINAL SELF-CORRECTION EXPERIMENT
# =========================================

question = "What are the risk factors for diabetes?"

bad_answer = """
Diabetes risk factors include obesity, family history,
physical inactivity, smoking, vitamin D deficiency,
and excessive salt consumption.
"""

# -----------------------------------------
# STEP 1: Retrieve evidence
# -----------------------------------------

documents = retrieve_documents(
    question,
    top_k=5
)

# -----------------------------------------
# STEP 2: Analyze initial answer
# -----------------------------------------

initial_analysis = analyze_answer(
    bad_answer,
    documents
)

# -----------------------------------------
# STEP 3: Calculate initial confidence
# -----------------------------------------

initial_confidence = calculate_confidence(
    question,
    initial_analysis["semantic_similarity"],
    initial_analysis["nli_entailment"],
    documents,
    bad_answer
)

print("=" * 70)
print("INITIAL ANSWER")
print("=" * 70)

print(bad_answer)

print(
    "Initial decision:",
    initial_analysis["decision"]
)

print(
    "Initial confidence:",
    round(
        initial_confidence["confidence_score"],
        4
    )
)

print(
    "Initial consistency:",
    round(
        initial_confidence["consistency"],
        4
    )
)

# -----------------------------------------
# STEP 4: Decide whether self-correction
# is required
# -----------------------------------------

should_correct = should_self_correct(
    initial_confidence["confidence_score"],
    initial_analysis["decision"]
)

print(
    "\nSelf-correction required:",
    should_correct
)

# -----------------------------------------
# STEP 5: Self-correction with retry
# -----------------------------------------

if should_correct:

    corrected_answer = self_correct_answer_with_retry(
        question,
        bad_answer,
        documents,
        max_retries=3
    )

else:

    corrected_answer = bad_answer


# -----------------------------------------
# STEP 6: Display corrected answer
# -----------------------------------------

print("\n" + "=" * 70)
print("CORRECTED ANSWER")
print("=" * 70)

print(corrected_answer)


# -----------------------------------------
# STEP 7: Re-verify corrected answer
# -----------------------------------------

corrected_analysis = analyze_answer(
    corrected_answer,
    documents
)

corrected_confidence = calculate_confidence(
    question,
    corrected_analysis["semantic_similarity"],
    corrected_analysis["nli_entailment"],
    documents,
    corrected_answer
)


# -----------------------------------------
# STEP 8: Final verification results
# -----------------------------------------

print("\n" + "=" * 70)
print("FINAL VERIFICATION")
print("=" * 70)

print(
    "Final decision:",
    corrected_analysis["decision"]
)

print(
    "Final confidence:",
    round(
        corrected_confidence["confidence_score"],
        4
    )
)

print(
    "Final consistency:",
    round(
        corrected_confidence["consistency"],
        4
    )
)


# -----------------------------------------
# STEP 9: Before vs After comparison
# -----------------------------------------

confidence_improvement = (
    corrected_confidence["confidence_score"]
    - initial_confidence["confidence_score"]
)

consistency_improvement = (
    corrected_confidence["consistency"]
    - initial_confidence["consistency"]
)

print("\n" + "=" * 70)
print("BEFORE vs AFTER")
print("=" * 70)

print(
    "Confidence improvement:",
    round(confidence_improvement, 4)
)

print(
    "Consistency improvement:",
    round(consistency_improvement, 4)
)

print(
    "Correction applied:",
    should_correct
)

Total claims detected: 6

Checking claim 1/6...

Checking claim 2/6...

Checking claim 3/6...

Checking claim 4/6...

Checking claim 5/6...

Checking claim 6/6...

Supported claims: 4/6
Consistency score: 0.6667
INITIAL ANSWER

Diabetes risk factors include obesity, family history,
physical inactivity, smoking, vitamin D deficiency,
and excessive salt consumption.

Initial decision: POTENTIAL HALLUCINATION
Initial confidence: 0.4691
Initial consistency: 0.6667

Self-correction required: True
Gemini temporarily unavailable. Retry 1/3...
Gemini temporarily unavailable. Retry 2/3...

CORRECTED ANSWER
Based on the provided evidence, the original answer contains unsupported claims. Smoking, vitamin D deficiency, and excessive salt consumption are not identified as risk factors for diabetes in the provided text. 

The supported risk factors for type 2 diabetes are:
* **Increasing age** [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
* **Overwe

In [99]:
for i, claim in enumerate(extract_claims(corrected_answer), start=1):

    hypothesis = claim_to_hypothesis(claim)

    claim_documents = retrieve_documents(
        hypothesis,
        top_k=5
    )

    combined_documents = retrieved_documents + claim_documents

    unique_documents = {
        doc["chunk_id"]: doc
        for doc in combined_documents
    }

    best_evidence = find_best_evidence(
        hypothesis,
        list(unique_documents.values()),
        top_n=5
    )

    print("\n" + "=" * 70)
    print(f"CLAIM {i}")
    print("=" * 70)
    print("Claim:", claim)
    print("Hypothesis:", hypothesis)

    if best_evidence is None:
        print("❌ No evidence found")
        continue

    print("Page:", best_evidence["page"])
    print("Similarity:", round(best_evidence["similarity"], 4))
    print("Entailment:", round(best_evidence["entailment"], 4))
    print("Contradiction:", round(best_evidence["contradiction"], 4))
    print("Evidence score:", round(best_evidence["evidence_score"], 4))

    if (
        best_evidence["entailment"] >= 0.70
        and best_evidence["contradiction"] < 0.50
        and best_evidence["evidence_score"] >= 0.60
    ):
        print("✅ SUPPORTED")
    else:
        print("❌ NOT SUPPORTED")


CLAIM 1
Claim: Increasing age
Hypothesis: Increasing age is a risk factor for type 2 diabetes.
Page: 9
Similarity: 0.6585
Entailment: 0.9954
Contradiction: 0.0001
Evidence score: 0.8269
✅ SUPPORTED

CLAIM 2
Claim: Overweight or obesity (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans)
Hypothesis: Overweight or obesity (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans) is a risk factor for type 2 diabetes.
Page: 9
Similarity: 0.7474
Entailment: 0.9902
Contradiction: 0.0002
Evidence score: 0.8687
✅ SUPPORTED

CLAIM 3
Claim: Family history of diabetes (parent or sibling)
Hypothesis: Family history of diabetes (parent or sibling) is a risk factor for type 2 diabetes.
Page: 9
Similarity: 0.5911
Entailment: 0.9919
Contradiction: 0.0003
Evidence score: 0.7914
✅ SUPPORTED

CLAIM 4
Claim: Physical inactivity
Hypothesis: Physical inactivity is a risk factor for type 2 diabetes.
Page: 9
Similarity: 0.6568
Entailment: 0.9921
Contradiction: 0.0002
Evidence score: 0.8244
✅ SUPPORTED

CLAIM 5
Cla

In [100]:
test_claims = [
    "Prediabetes (higher than normal blood glucose levels) is a risk factor for type 2 diabetes.",
    "Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day) are emerging risk factors for type 2 diabetes."
]

for claim in test_claims:

    print("\n" + "=" * 70)
    print("CLAIM:", claim)
    print("=" * 70)

    docs = retrieve_documents(claim, top_k=5)

    best = find_best_evidence(
        claim,
        docs,
        top_n=5
    )

    if best is None:
        print("❌ No evidence found")
        continue

    print("Page:", best["page"])
    print("Similarity:", round(best["similarity"], 4))
    print("Entailment:", round(best["entailment"], 4))
    print("Neutral:", round(best["neutral"], 4))
    print("Contradiction:", round(best["contradiction"], 4))
    print("Evidence score:", round(best["evidence_score"], 4))


CLAIM: Prediabetes (higher than normal blood glucose levels) is a risk factor for type 2 diabetes.
Page: 15
Similarity: 0.6614
Entailment: 0.0016
Neutral: 0.9984
Contradiction: 0.0
Evidence score: 0.3315

CLAIM: Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day) are emerging risk factors for type 2 diabetes.
Page: 9
Similarity: 0.8047
Entailment: 0.0091
Neutral: 0.9901
Contradiction: 0.0008
Evidence score: 0.4068


In [101]:
test_claims = [
    "Prediabetes (higher than normal blood glucose levels) is a risk factor for type 2 diabetes.",
    "Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day) are emerging risk factors for type 2 diabetes."
]

for claim in test_claims:

    print("\n" + "=" * 80)
    print("CLAIM:", claim)
    print("=" * 80)

    docs = retrieve_documents(claim, top_k=3)

    for rank, doc in enumerate(docs, start=1):
        print(f"\n--- Retrieved Evidence {rank} | Page {doc['page']} ---")
        print(doc["text"][:1500])


CLAIM: Prediabetes (higher than normal blood glucose levels) is a risk factor for type 2 diabetes.

--- Retrieved Evidence 1 | Page 15 ---
15
PRINCIPLE 2:
Manage Prediabetes to Prevent or Delay the Onset of Type 2 Diabetes
Progression to type 2 diabetes among people with prediabetes is not inevitable. Modest, sustained 
weight loss, increased physical activity, and/or metformin therapy in these individuals can prevent 
or delay the onset of type 2 diabetes.
The National Institutes of Health-led Diabetes Prevention Program (DPP)1 and the Finnish Diabetes 
Prevention Program2 aimed for and achieved a mean weight loss of 7 percent and 5 percent, 
respectively, in study participants randomized to the lifestyle intervention. In both studies, the 
lifestyle intervention, compared with placebo, reduced the incidence of diabetes by 58 percent 
over 3 years. In the DPP, these results were similar in all groups, including men and women, all 
racial and ethnic groups, as well as in women with a 

In [102]:
test_pairs = [
    (
        "Because persons with these glucose levels are at increased risk of developing type 2 diabetes, this condition is termed prediabetes by the Centers for Disease Control and Prevention (CDC) and other organizations.",
        "Prediabetes is associated with an increased risk of developing type 2 diabetes."
    ),
    (
        "Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day) are emerging risk factors.",
        "Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day) are emerging risk factors for type 2 diabetes."
    )
]

for premise, hypothesis in test_pairs:

    entailment, neutral, contradiction = nli_score(
        premise,
        hypothesis
    )

    print("\n" + "=" * 70)
    print("Premise:", premise)
    print("Hypothesis:", hypothesis)
    print("Entailment:", round(entailment, 4))
    print("Neutral:", round(neutral, 4))
    print("Contradiction:", round(contradiction, 4))


Premise: Because persons with these glucose levels are at increased risk of developing type 2 diabetes, this condition is termed prediabetes by the Centers for Disease Control and Prevention (CDC) and other organizations.
Hypothesis: Prediabetes is associated with an increased risk of developing type 2 diabetes.
Entailment: 0.9975
Neutral: 0.0024
Contradiction: 0.0001

Premise: Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day) are emerging risk factors.
Hypothesis: Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day) are emerging risk factors for type 2 diabetes.
Entailment: 0.0001
Neutral: 0.9994
Contradiction: 0.0005


In [103]:
page9_doc = next(
    doc for doc in retrieve_documents(
        "Obstructive sleep apnea and chronic sleep deprivation risk factors for type 2 diabetes",
        top_k=5
    )
    if doc["page"] == 9
)

premise = page9_doc["text"]

hypothesis = (
    "Obstructive sleep apnea and chronic sleep deprivation "
    "(< 6 hours/day) are emerging risk factors for type 2 diabetes."
)

entailment, neutral, contradiction = nli_score(
    premise,
    hypothesis
)

print("Entailment:", round(entailment, 4))
print("Neutral:", round(neutral, 4))
print("Contradiction:", round(contradiction, 4))

Entailment: 0.0091
Neutral: 0.9901
Contradiction: 0.0008


In [104]:
def is_claim_supported(best_evidence):
    """
    Decide whether an individual claim is supported by retrieved evidence.

    Primary signal:
    - NLI entailment

    Bounded semantic fallback:
    - Strong semantic similarity
    - Very low contradiction
    - Sufficient overall evidence score
    """

    if best_evidence is None:
        return False

    similarity = best_evidence["similarity"]
    entailment = best_evidence["entailment"]
    contradiction = best_evidence["contradiction"]
    evidence_score = best_evidence["evidence_score"]

    # Strong NLI evidence
    if (
        entailment >= 0.70
        and contradiction < 0.50
        and evidence_score >= 0.60
    ):
        return True

    # Bounded semantic fallback
    # Used when NLI predicts "neutral" despite strong evidence alignment.
    if (
        similarity >= 0.75
        and contradiction < 0.10
        and evidence_score >= 0.40
    ):
        return True

    return False

In [105]:
def calculate_consistency(answer, retrieved_documents):

    claims = extract_claims(answer)

    print("Total claims detected:", len(claims))

    if not claims:
        return 0.0

    supported_claims = 0

    for i, claim in enumerate(claims, start=1):

        print(f"\nChecking claim {i}/{len(claims)}...")

        hypothesis = claim_to_hypothesis(claim)

        claim_documents = retrieve_documents(
            hypothesis,
            top_k=5
        )

        # Combine original retrieval + claim-specific retrieval
        combined_documents = (
            retrieved_documents + claim_documents
        )

        # Remove duplicate chunks
        unique_documents = {
            doc["chunk_id"]: doc
            for doc in combined_documents
        }

        best_evidence = find_best_evidence(
            hypothesis,
            list(unique_documents.values()),
            top_n=5
        )

        if best_evidence is None:
            print("❌ No evidence")
            continue

        if is_claim_supported(best_evidence):
            supported_claims += 1
            print("✅ SUPPORTED")
        else:
            print("❌ NOT SUPPORTED")

    consistency_score = supported_claims / len(claims)

    print(
        f"\nSupported claims: "
        f"{supported_claims}/{len(claims)}"
    )

    print(
        "Consistency score:",
        round(consistency_score, 4)
    )

    return float(consistency_score)

In [106]:
final_consistency = calculate_consistency(
    corrected_answer,
    documents
)

print(
    "\nFinal consistency:",
    round(final_consistency, 4)
)

Total claims detected: 9

Checking claim 1/9...
✅ SUPPORTED

Checking claim 2/9...
✅ SUPPORTED

Checking claim 3/9...
✅ SUPPORTED

Checking claim 4/9...
✅ SUPPORTED

Checking claim 5/9...
❌ NOT SUPPORTED

Checking claim 6/9...
✅ SUPPORTED

Checking claim 7/9...
✅ SUPPORTED

Checking claim 8/9...
✅ SUPPORTED

Checking claim 9/9...
✅ SUPPORTED

Supported claims: 8/9
Consistency score: 0.8889

Final consistency: 0.8889


In [107]:
final_question = "What are the risk factors for diabetes?"

initial_answer = """
Diabetes risk factors include obesity, family history,
physical inactivity, smoking, vitamin D deficiency,
and excessive salt consumption.
"""

final_documents = retrieve_documents(
    final_question,
    top_k=5
)

print("=" * 70)
print("INITIAL ANSWER VERIFICATION")
print("=" * 70)

initial_analysis = analyze_answer(
    initial_answer,
    final_documents
)

initial_confidence = calculate_confidence(
    final_question,
    initial_analysis["semantic_similarity"],
    initial_analysis["nli_entailment"],
    final_documents,
    initial_answer
)

print("Initial decision:", initial_analysis["decision"])
print("Initial confidence:",
      round(initial_confidence["confidence_score"], 4))

print("\n" + "=" * 70)
print("SELF-CORRECTION")
print("=" * 70)

corrected_answer = self_correct_answer_with_retry(
    final_question,
    initial_answer,
    final_documents
)

print("\nCorrected Answer:\n")
print(corrected_answer)

print("\n" + "=" * 70)
print("FINAL VERIFICATION")
print("=" * 70)

final_analysis = analyze_answer(
    corrected_answer,
    final_documents
)

final_confidence = calculate_confidence(
    final_question,
    final_analysis["semantic_similarity"],
    final_analysis["nli_entailment"],
    final_documents,
    corrected_answer
)

final_consistency = calculate_consistency(
    corrected_answer,
    final_documents
)

print("\nFinal decision:", final_analysis["decision"])
print("Final confidence:",
      round(final_confidence["confidence_score"], 4))
print("Final consistency:",
      round(final_consistency, 4))

INITIAL ANSWER VERIFICATION
Total claims detected: 1

Checking claim 1/1...
✅ SUPPORTED

Supported claims: 1/1
Consistency score: 1.0
Initial decision: POTENTIAL HALLUCINATION
Initial confidence: 0.5358

SELF-CORRECTION
Gemini temporarily unavailable. Retry 1/3...
Gemini temporarily unavailable. Retry 2/3...
Gemini temporarily unavailable. Retry 3/3...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [108]:
corrected_answer = self_correct_answer_with_retry(
    final_question,
    initial_answer,
    final_documents
)

print("\nCorrected Answer:\n")
print(corrected_answer)

Gemini temporarily unavailable. Retry 1/3...

Corrected Answer:

Based on the provided evidence, risk factors for type 2 diabetes include:

* **Overweight or obesity** (BMI ≥ 25 kg/m², or ≥ 23 kg/m² for Asian Americans) [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
* **Increasing age** [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
* **Family history of diabetes** (parent or sibling) [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
* **Physical inactivity** [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 9]
* **Prediabetes** (elevated blood glucose levels higher than normal) [Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 5]
* **Membership in a high-risk population** (African American, Hispanic/Latino, American Indian, Alaska Native, Asian American,

In [109]:
final_analysis = analyze_answer(
    corrected_answer,
    final_documents
)

final_confidence = calculate_confidence(
    final_question,
    final_analysis["semantic_similarity"],
    final_analysis["nli_entailment"],
    final_documents,
    corrected_answer
)

final_consistency = calculate_consistency(
    corrected_answer,
    final_documents
)

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print("Final decision:", final_analysis["decision"])
print("Final confidence:",
      round(final_confidence["confidence_score"], 4))
print("Final consistency:",
      round(final_consistency, 4))

Total claims detected: 9

Checking claim 1/9...
✅ SUPPORTED

Checking claim 2/9...
✅ SUPPORTED

Checking claim 3/9...
✅ SUPPORTED

Checking claim 4/9...
✅ SUPPORTED

Checking claim 5/9...
❌ NOT SUPPORTED

Checking claim 6/9...
✅ SUPPORTED

Checking claim 7/9...
✅ SUPPORTED

Checking claim 8/9...
✅ SUPPORTED

Checking claim 9/9...
✅ SUPPORTED

Supported claims: 8/9
Consistency score: 0.8889
Total claims detected: 9

Checking claim 1/9...
✅ SUPPORTED

Checking claim 2/9...
✅ SUPPORTED

Checking claim 3/9...
✅ SUPPORTED

Checking claim 4/9...
✅ SUPPORTED

Checking claim 5/9...
❌ NOT SUPPORTED

Checking claim 6/9...
✅ SUPPORTED

Checking claim 7/9...
✅ SUPPORTED

Checking claim 8/9...
✅ SUPPORTED

Checking claim 9/9...
✅ SUPPORTED

Supported claims: 8/9
Consistency score: 0.8889

FINAL RESULTS
Final decision: CONTRADICTED
Final confidence: 0.819
Final consistency: 0.8889


## Final Results and Observations

The self-correction pipeline was evaluated using a deliberately
hallucinated diabetes risk-factor answer.

### Observations

- The initial answer contained unsupported risk factors.
- The hallucination detection module identified the answer as unsafe.
- The self-correction module generated a more evidence-grounded answer.
- Claim-level verification evaluated the corrected answer against
  retrieved evidence.
- The corrected answer achieved a consistency score of 0.8889
  (8/9 supported claims) under the current verification pipeline.
- One claim remained unsupported by the current verifier and was not
  artificially marked as supported.
- This demonstrates that semantic retrieval, NLI-based verification,
  confidence scoring, and self-correction can be combined into a
  closed-loop medical RAG pipeline.

### Pipeline

Question
→ Document Retrieval
→ LLM Answer Generation
→ Hallucination Detection
→ Confidence Scoring
→ Claim Extraction
→ Evidence Verification
→ Self-Correction
→ Final Verification